# Análisis Cuantitativo - Tesis de Maestría

## Análisis de percepciones y condiciones de adopción en la industria del software

Agustín José Bobba – 156448

Juan Ignacio Santiago – 173621

Guillermo Zooby – 149807

**Datos:** 193 respuestas válidas del sector tecnológico/software  
**Roles:** Colaboradores (93), Líderes (50), Dirección/Gerencia (29), RRHH (21)

---
## 1. Configuración y Carga de Datos

In [ ]:
# =============================================================================
# CONFIGURACIÓN INICIAL
# =============================================================================

# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Configuración de visualización
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 100

# Paleta de colores profesional
COLORES_PRINCIPALES = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
COLORES_LIKERT = ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']
COLORES_ROLES = {'Colaborador': '#2E86AB', 'Líder': '#A23B72', 'Dirección/Gerencia': '#F18F01', 'RRHH': '#C73E1D'}

# Configuración de pandas para mostrar más columnas
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Librerías cargadas correctamente")
print(f"  - Pandas: {pd.__version__}")
print(f"  - NumPy: {np.__version__}")

In [ ]:
# =============================================================================
# CARGA DE DATOS (ARCHIVO YA FILTRADO)
# =============================================================================
# El archivo Cuestionario_Tesis_Filtrado.xlsx ya tiene aplicados los filtros:
# 1. Respuestas desde el 13 de noviembre de 2025 (fecha de publicación oficial)
# 2. Solo empresas del sector tecnológico/software
# 3. Solo participantes que aceptaron participar de forma anónima

df_raw = pd.read_excel('Cuestionario_Tesis_Filtrado.xlsx')

print(f"Datos cargados exitosamente")
print(f"  - Total de respuestas: {df_raw.shape[0]}")
print(f"  - Total de columnas: {df_raw.shape[1]}")
print(f"\n Nota: Este archivo ya está filtrado por fecha, sector tech y aceptación")

In [ ]:
# =============================================================================
# VERIFICACIÓN DE DATOS CARGADOS
# =============================================================================

print("="*70)
print("VERIFICACIÓN DE DATOS FILTRADOS")
print("="*70)

# Verificar distribución por rol
col_rol = 'Rol actual'
if col_rol in df_raw.columns:
    print(f"\nDistribución por Rol:")
    print(df_raw[col_rol].value_counts())

# Verificar rango de fechas
col_timestamp = 'Marca temporal'
if col_timestamp in df_raw.columns:
    df_raw[col_timestamp] = pd.to_datetime(df_raw[col_timestamp])
    print(f"\nRango de fechas:")
    print(f"   - Primera respuesta: {df_raw[col_timestamp].min()}")
    print(f"   - Última respuesta: {df_raw[col_timestamp].max()}")

print(f"\n {len(df_raw)} respuestas válidas listas para análisis")

In [ ]:
# =============================================================================
# RENOMBRAR COLUMNAS PARA FACILITAR EL ANÁLISIS
# =============================================================================

# Diccionario de nombres cortos para las columnas
nombres_columnas = {
    # === COLUMNAS GENERALES (todos los encuestados) ===
    'Marca temporal': 'timestamp',
    'Confirmo conocer de que consta la encuesta y acepto participar de forma anónima': 'acepta_participar',
    'Soy nacido en Uruguay': 'nacido_uruguay',
    'Edad': 'edad',
    'Me identifico como': 'genero',
    'La empresa en la cual trabajo tiene a nivel global': 'tamanio_empresa_global',
    'Mi antigüedad en la empresa actual es': 'antiguedad',
    'La empresa en la cual trabajo tiene una trayectoria en la industria': 'trayectoria_empresa',
    'Mi carga horaria de trabajo es': 'carga_horaria',
    'Mi modalidad de trabajo es': 'modalidad_trabajo',
    'En la empresa que trabajo cuento con oportunidades claras de desarrollo profesional': 'oportunidades_desarrollo',
    'Recibo feedback útil y frecuente sobre mi desempeño laboral': 'feedback_desempenio',
    'Mi carga de trabajo es razonable y no me resulta excesiva ': 'carga_razonable',
    'Mi compensación total (salario + beneficios) es satisfactoria y competitiva': 'compensacion_satisfactoria',
    'La cultura de la empresa y de mi equipo promueve colaboración y confianza': 'cultura_colaborativa',
    'Mi líder se preocupa por mi y fomenta mi crecimiento y bienestar': 'lider_preocupa',
    'Es probable que busque nuevas oportunidades laborales en los próximos meses': 'buscar_oportunidades',
    'Que tan familiarizado estas con el concepto de IA?': 'familiaridad_ia',
    'Crees que el concepto general de la IA es algo positivo para nuestra realidad actual?': 'ia_positiva',
    'Cuán frecuentemente utilizas herramientas de IA en tus tareas cotidianas?': 'frecuencia_uso_ia',
    'Cuán estimulado es el uso de la IA en la empresa donde trabajas?': 'estimulo_ia_empresa',
    'Cuán confiable crees que es la IA al utilizarla para desarrollar tus tareas?': 'confianza_ia',
    'Utilizo la IA para apoyarme en mis tareas cotidianas como ser:': 'usos_ia',
    'Trabajo en el sector de RRHH': 'trabaja_rrhh',
    'Rol actual': 'rol',
    
    # === PREGUNTAS PARA COLABORADORES (48.2% de respuestas) ===
    'Estoy de acuerdo en que se usen sistemas de IA que analicen datos como mis horas registradas, entregas de tareas, participación en reuniones y cumplimiento de plazos para detectar si podría estar desmotivado o pensando en renunciar  ': 'col_ia_detectar_desmotivacion',
    'Estoy de acuerdo en que un sistema de IA analice mis habilidades según proyectos en los que trabajé, evaluaciones de desempeño y cursos completados para recomendarme nuevas capacitaciones relevantes para mi crecimiento  ': 'col_ia_capacitacion',
    'Estoy de acuerdo en que un sistema de IA analice mi perfil profesional, intereses declarados y desempeño para sugerirme posibles cambios de rol o crecimiento dentro de la empresa  ': 'col_ia_carrera',
    'Estoy de acuerdo en que se utilicen herramientas de IA que monitoreen indicadores como horas extra o mensajes enviados fuera de horario para ofrecerme recomendaciones automáticas de equilibrio personal/laboral o descanso  ': 'col_ia_equilibrio_personal',
    'Estoy de acuerdo en que a partir de encuestas de bienestar, registros médicos voluntarios o uso de licencias por estrés, un sistema de IA me sugiera apoyo emocional o recursos de ayuda psicológica confidencial  ': 'col_ia_apoyo_emocional',
    'Estoy de acuerdo en que un sistema de IA analice mis interacciones con compañeros mediante datos de chats, correos o llamadas para determinar si estoy aislado o perdiendo conexión con el equipo  ': 'col_ia_interacciones',
    'Estoy de acuerdo en que un asistente virtual inteligente responda mis consultas sobre vacaciones, recibos o políticas internas usando información de mis registros de RRHH sin necesidad de esperar respuesta humana ': 'col_ia_asistente_admin',
    'Estoy de acuerdo en que se utilicen herramientas que analicen mi tono de voz, expresiones faciales en videollamadas o palabras utilizadas en mensajes para interpretar mis emociones o nivel de satisfacción ': 'col_ia_emociones',
    'Estaría abierto a tratar cuestiones personales y laborales con un asistente virtual inteligente de la empresa para obtener orientaciones o sugerencias ': 'col_ia_asistente_personal',
    'Estoy de acuerdo en que se utilicen estas tecnologías de IA solo si se me informa claramente qué datos se recopilan, cómo se analizan y para qué se van a usar  ': 'col_ia_transparencia',
    
    # === PREGUNTAS PARA LÍDERES (25.9% de respuestas) ===
    'Me parecería útil contar con herramientas de IA que analicen datos de asistencia, cumplimiento de tareas y participación en reuniones de mi equipo para detectar señales tempranas de desmotivación  ': 'lid_ia_detectar_desmotivacion',
    'Estoy de acuerdo en que un sistema de IA genere recomendaciones de capacitación para cada integrante del equipo, basadas en sus proyectos realizados, evaluaciones de desempeño y habilidades técnicas  ': 'lid_ia_capacitacion',
    'Estoy de acuerdo que un sistema de IA sugiera posibles movimientos internos o cambios de rol cuando detecta que un colaborador tiene habilidades que podrían aprovecharse mejor en otras áreas  ': 'lid_ia_movilidad',
    'Me resultaría útil recibir alertas automáticas cuando la IA identifica sobrecarga de trabajo en algún integrante del equipo, basándose en horas extras, mensajes enviados fuera de horario o disminución en la productividad  ': 'lid_ia_alertas_sobrecarga',
    'Considero que el monitoreo y análisis de la comunicación de los miembros del equipo (frecuencia de reuniones, colaboración en tareas, mensajes) puede ayudar a detectar aislamiento o falta de integración  ': 'lid_ia_comunicacion',
    'Estoy de acuerdo en que un agente virtual inteligente que responda consultas de mi equipo sobre políticas, licencias, capacitaciones o evaluaciones ayudaría a reducir mi carga operativa diaria ': 'lid_ia_agente_admin',
    'Me incomodaría que se usen herramientas que, basadas solo en datos de productividad o comparaciones estadísticas, cuestionen decisiones humanas de liderazgo  ': 'lid_incomodidad_decisiones_auto',
    'Pienso que los miembros de mi equipo podrían sentirse igual de cómodos conversando sobre asuntos personales o laborales con un asistente virtual inteligente que conmigo ': 'lid_equipo_comodo_asistente',
    'Estaría dispuesto a aplicar estas herramientas en mi equipo siempre que se respeten la privacidad, el consentimiento y que las decisiones finales sigan siendo humanas ': 'lid_disposicion_con_etica',
    'Creo que el uso de sistemas de IA que analicen expresiones faciales o tonos de voz durante reuniones virtuales podría generar preocupación en mi equipo': 'lid_ia_expresiones_preocupacion',
    
    # === PREGUNTAS PARA GERENCIA parte 1 (15% de respuestas) ===
    'Estoy de acuerdo en que se empleen sistemas de IA que analicen datos de desempeño, rotación, ausentismo y clima laboral para anticipar riesgos de fuga de talento clave  ': 'dir_ger_ia_fuga_talento',
    'Creo que sería importante invertir en sistemas de IA que identifiquen habilidades presentes en la organización y propongan planes de formación personalizados según el perfil de cada empleado y necesidades futuras del negocio': 'dir_ger_ia_formacion_personalizada',
    'Confiaría en que un sistema de IA, frente a vacantes, sugiera reemplazos de personal basados en datos de experiencia, evaluaciones y trayectoria dentro de la empresa ': 'dir_ger_ia_sucesion',
    'Consideraría valioso contar con un sistema de IA que pueda detectar señales tempranas de agotamiento o estrés en empleados analizando horas trabajadas, uso de licencias, reducción de productividad o mensajes enviados fuera del horario laboral ': 'dir_ger_ia_deteccion_agotamiento',
    'Estoy de acuerdo en que analizar patrones de comunicación entre colaboradores y áreas (reuniones, correos, proyectos) mediante un sistema de IA puede ayudar a identificar equipos aislados o colaboradores desconectados de la organización ': 'dir_ger_ia_patrones_comunicacion',
    'Me parecería eficiente que algunas consultas administrativas de los colaboradores se gestionen mediante asistentes virtuales inteligentes que acceden a datos de RRHH, evitando demoras y sobrecarga de tareas ': 'dir_ger_ia_asistente_consultas',
    'Estoy de acuerdo en que se utilicen sistemas de IA que interpreten emociones o estados de ánimo mediante análisis de voz, expresiones faciales o escritura siempre y cuando sea previamente comunicado a los colaboradores ': 'dir_ger_ia_interpretar_emociones',
    'Consideraría valioso que un sistema de IA me brinde recomendaciones sobre ajustes salariales para los colaboradores de mi empresa, a partir de un análisis de mercado, del rendimiento individual y la equidad interna': 'dir_ger_ia_compensaciones',
    'Me preocupa que la implementación de estas tecnologías basadas en IA sean percibidas como una forma de control o vigilancia a pesar de haber comunicado adecuadamente su propósito, alcances y límites ': 'dir_ger_preocupa_vigilancia',
    'Estoy de acuerdo en que estas herramientas podrían aportar valor a la organización, siempre que se apliquen gradualmente y con políticas claras de privacidad': 'dir_ger_valor_con_politicas',
    
    # === PREGUNTAS PARA RRHH parte 2 (10.9% de respuestas) ===
    'Considero útil que la IA analice datos históricos de desempeño, rotación, ausentismo y encuestas de clima laboral para identificar áreas o perfiles con riesgo de desvinculación  ': 'rrhh_ia_historicos_retencion',
    'Me resultaría valioso contar con herramientas basadas en IA que identifiquen brechas de habilidades analizando descripciones de puestos, evaluaciones de desempeño, certificaciones y cursos completados para proponer planes de formación personalizados ': 'rrhh_ia_brechas_habilidades',
    'Estoy de acuerdo que sería beneficioso contar con sistemas basados en IA que sugieran posibles movimientos internos o reemplazos para vacantes basándose en experiencia, habilidades registradas y trayectoria dentro de la empresa ': 'rrhh_ia_movimientos_internos',
    'Considero útil que la IA detecte señales y notifique a RRHH acerca del desgaste emocional de los colaboradores, analizando horas extra, disminución de productividad, uso de licencias médicas o comentarios en encuestas de bienestar ': 'rrhh_ia_desgaste_emocional',
    'Estoy de acuerdo en que contar con herramientas de IA que analicen lenguaje en encuestas, foros internos o formularios de feedback permitiría detectar emociones negativas o falta de compromiso antes de que se manifieste explícitamente ': 'rrhh_ia_analisis_lenguaje',
    'Estoy de acuerdo en que un sistema de IA que monitoree redes de colaboración (frecuencia de reuniones, mensajes entre áreas, participación en proyectos) puede ayudar a identificar equipos aislados o sobrecargados': 'rrhh_ia_redes_colaboracion',
    'Estoy de acuerdo en que la implementación de un asistente virtual inteligente que responda preguntas sobre licencias, vacaciones, recibos, políticas o capacitaciones accediendo a datos del sistema de RRHH en tiempo real contribuiría a alivianar mis tareas diarias': 'rrhh_ia_asistente_eficiencia',
    'Estoy de acuerdo en que un sistema de IA realice recomendaciones de ajustes salariales o beneficios basadas en análisis de mercado, desempeño individual y equidad interna': 'rrhh_ia_recomendaciones_salario',
    'Creo que la inclusión de estas herramientas basadas en IA pueden generar desconfianza entre los colaboradores, a pesar de comunicar claramente qué datos se recopilan, cómo se analizan y quién accede a los resultados ': 'rrhh_desconfianza_vigilancia',
    'Considero necesario que, antes de implementar estas soluciones basadas en IA, se establezcan políticas claras de privacidad, uso responsable de datos y consentimiento informado por parte de los colaboradores': 'rrhh_politicas_claras',
    
    # === COLUMNAS FINALES (todos los encuestados) ===
    'La empresa en la cual trabajo es': 'tipo_empresa',
    'Más allá de su valor potencial, considero que el uso de herramientas de IA podría resultar invasivo o generar preocupación respecto a la privacidad de las personas': 'ia_invasivo',
    'Creo que las herramientas de IA pueden aportar valor, pero la cultura organizacional de la empresa en la que trabajo aún no está preparada para su implementación plena.': 'cultura_no_preparada',
    'En lo personal, desde mi rol impulsaría la incorporación de sistemas de IA para la gestión de RRHH en la empresa donde trabajo': 'impulsaria_ia',
    'Consideras que hay otra tecnología o enfoque que podría aportar valor a la gestión y retención del talento?\n\nPor favor descríbela brevemente': 'otra_tecnologia',
    'Confirmo que mi empresa está vinculada al sector tecnológico o a la industria del software': 'empresa_tech'
}

# Crear copia del dataframe y renombrar columnas que existan
df = df_raw.copy()
columnas_existentes = {k: v for k, v in nombres_columnas.items() if k in df.columns}
df = df.rename(columns=columnas_existentes)

print(f" {len(columnas_existentes)} columnas renombradas de {len(df.columns)} totales")

# Mostrar resumen de columnas por sección
print(f"\n Resumen de columnas renombradas:")
print(f"   - Generales: {len([c for c in columnas_existentes.values() if not c.startswith(('col_', 'lid_', 'dir_ger_'))])}")
print(f"   - Colaboradores (col_*): {len([c for c in columnas_existentes.values() if c.startswith('col_')])}")
print(f"   - Líderes (lid_*): {len([c for c in columnas_existentes.values() if c.startswith('lid_')])}")
print(f"   - Direccion (dir_ger_*): {len([c for c in columnas_existentes.values() if c.startswith('dir_ger_')])}")

In [ ]:
# =============================================================================
# LIMPIEZA Y PREPARACIÓN DE DATOS
# =============================================================================

# 1. Crear columna 'rol_final' combinando 'rol' y 'trabaja_rrhh'
# Los que trabajan en RRHH tienen rol=NaN, su rol es RRHH

def asignar_rol(row):
    # Si trabaja en RRHH, asignar rol RRHH
    if row.get('trabaja_rrhh') == 'Si':
        return 'RRHH'
    
    # Si tiene rol definido, normalizar
    rol_original = row.get('rol')
    if pd.notna(rol_original):
        rol_str = str(rol_original)
        if 'Colaborador' in rol_str:
            return 'Colaborador'
        elif 'Líder' in rol_str:
            return 'Líder'
        elif 'Dirección' in rol_str or 'gerencia' in rol_str.lower():
            return 'Dirección/Gerencia'
        return rol_str
    
    return np.nan

# Aplicar función para crear rol normalizado
df['rol'] = df.apply(asignar_rol, axis=1)

# 2. Crear categorías ordenadas para variables ordinales
orden_edad = ['Menos de 25 años', 'Entre 25 y 35 años', 'Entre 36 y 45 años', 
              'Entre 46 y 55 años', 'Mayor de 55 años']

orden_antiguedad = ['De 0 a 2 años', 'De 2 a 5 años', 'De 5 a 15 años', 'Más de 15 años']

orden_tamanio = ['Menos de 50 empleados', 'De 50 a 100 empleados', 
                 'De 100 a 300 empleados', 'Más de 300 empleados']

# 3. Convertir a categorías ordenadas
if 'edad' in df.columns:
    df['edad'] = pd.Categorical(df['edad'], categories=orden_edad, ordered=True)

if 'antiguedad' in df.columns:
    df['antiguedad'] = pd.Categorical(df['antiguedad'], categories=orden_antiguedad, ordered=True)

if 'tamanio_empresa_global' in df.columns:
    df['tamanio_empresa_global'] = pd.Categorical(df['tamanio_empresa_global'], 
                                                   categories=orden_tamanio, ordered=True)

# 4. Filtrar respuestas válidas (que tienen rol definido)
df_valido = df[df['rol'].notna()].copy()

print(f" Datos limpiados y preparados")
print(f"  - Respuestas válidas para análisis: {len(df_valido)} de {len(df)}")
print(f"\n Distribución por Rol:")
print(df_valido['rol'].value_counts().sort_values(ascending=False))

In [ ]:
# =============================================================================
# VERIFICAR ESTRUCTURA FINAL
# =============================================================================

print(" Distribución de roles (datos válidos):")
print(df_valido['rol'].value_counts())

print(f"\n Total de respuestas válidas: {len(df_valido)}")
print(f"\n Columnas disponibles: {len(df_valido.columns)}")

---
## 2. Perfil de Encuestados

In [ ]:
# =============================================================================
# CREAR DIRECTORIO PARA GRÁFICOS
# =============================================================================

import os
os.makedirs('graficos', exist_ok=True)
print(" Directorio 'graficos' creado/verificado")

In [ ]:
# =============================================================================
# TABLA RESUMEN: PERFIL DEMOGRÁFICO
# =============================================================================

def crear_tabla_frecuencias(df, columna, nombre_variable):
    """Crea tabla de frecuencias con conteo y porcentaje"""
    freq = df[columna].value_counts(dropna=False)
    pct = (freq / len(df) * 100).round(1)
    tabla = pd.DataFrame({
        nombre_variable: freq.index,
        'n': freq.values,
        '%': pct.values
    })
    return tabla

# Crear tablas de perfil
print("="*70)
print("PERFIL DEMOGRÁFICO Y PROFESIONAL DE ENCUESTADOS")
print("="*70)

variables_perfil = [
    ('rol', 'Rol'),
    ('edad', 'Grupo Etario'),
    ('genero', 'Género'),
    ('antiguedad', 'Antigüedad'),
    ('tamanio_empresa_global', 'Tamaño de Empresa'),
    ('modalidad_trabajo', 'Modalidad de Trabajo'),
    ('tipo_empresa', 'Tipo de Empresa'),
    ('trabaja_rrhh', 'Trabaja en RRHH')
]

for col, nombre in variables_perfil:
    if col in df_valido.columns:
        print(f"\n--- {nombre} ---")
        tabla = crear_tabla_frecuencias(df_valido, col, nombre)
        print(tabla.to_string(index=False))

In [ ]:
# =============================================================================
# GRÁFICO: DISTRIBUCIÓN POR ROL
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Distribución de los 4 roles
rol_counts = df_valido['rol'].value_counts()

colors = [COLORES_ROLES.get(r, '#888888') for r in rol_counts.index]

bars = ax.barh(rol_counts.index, rol_counts.values, color=colors)
ax.set_xlabel('Número de respuestas')
ax.set_title('Distribución por Rol', fontweight='bold', fontsize=14)

# Añadir etiquetas con valores y porcentajes
for bar, val in zip(bars, rol_counts.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, 
             f'{val} ({val/len(df_valido)*100:.1f}%)', 
             va='center', fontsize=11)

plt.tight_layout()
plt.savefig('graficos/02_distribucion_roles.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n Total: {len(df_valido)} respuestas válidas")

In [ ]:
# =============================================================================
# GRÁFICO: PERFIL DEMOGRÁFICO COMPLETO
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Edad
ax = axes[0, 0]
edad_counts = df_valido['edad'].value_counts().sort_index()
edad_counts.plot(kind='bar', ax=ax, color=COLORES_PRINCIPALES[0], edgecolor='white')
ax.set_title('Distribución por Edad', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Frecuencia')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(edad_counts.values):
    ax.text(i, v + 1, f'{v}', ha='center', fontsize=9)

# 2. Género
ax = axes[0, 1]
# Agrupar en Hombre, Mujer y Otros
genero_raw = df_valido['genero'].value_counts()
genero_agrupado = {}
for cat, val in genero_raw.items():
    cat_lower = str(cat).lower()
    if 'hombre' in cat_lower or 'masculino' in cat_lower:
        genero_agrupado['Hombre'] = genero_agrupado.get('Hombre', 0) + val
    elif 'mujer' in cat_lower or 'femenino' in cat_lower:
        genero_agrupado['Mujer'] = genero_agrupado.get('Mujer', 0) + val
    else:
        genero_agrupado['Otros'] = genero_agrupado.get('Otros', 0) + val
genero_counts = pd.Series(genero_agrupado)
colors_gen = ['#2E86AB', '#A23B72', '#F18F01']
wedges, texts, autotexts = ax.pie(
    genero_counts.values,
    labels=None,
    autopct='%1.1f%%',
    colors=colors_gen[:len(genero_counts)],
    explode=[0.03] * len(genero_counts),
    startangle=90,
    pctdistance=0.72
)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')
    autotext.set_color('black')
ax.legend(
    wedges,
    [f'{label} (n={val})' for label, val in zip(genero_counts.index, genero_counts.values)],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=3,
    fontsize=9
)
ax.set_title('Distribución por Género', fontweight='bold')

# 3. Antigüedad
ax = axes[0, 2]
ant_counts = df_valido['antiguedad'].value_counts().sort_index()
ant_counts.plot(kind='bar', ax=ax, color=COLORES_PRINCIPALES[1], edgecolor='white')
ax.set_title('Distribución por Antigüedad', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Frecuencia')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(ant_counts.values):
    ax.text(i, v + 1, f'{v}', ha='center', fontsize=9)

# 4. Tamaño de empresa
ax = axes[1, 0]
tam_counts = df_valido['tamanio_empresa_global'].value_counts().sort_index()
tam_counts.plot(kind='bar', ax=ax, color=COLORES_PRINCIPALES[2], edgecolor='white')
ax.set_title('Distribución por Tamaño de Empresa', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Frecuencia')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(tam_counts.values):
    ax.text(i, v + 1, f'{v}', ha='center', fontsize=9)

# 5. Modalidad de trabajo
ax = axes[1, 1]
mod_counts = df_valido['modalidad_trabajo'].value_counts()
mod_counts.plot(kind='bar', ax=ax, color=COLORES_PRINCIPALES[3], edgecolor='white')
ax.set_title('Distribución por Modalidad de Trabajo', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Frecuencia')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(mod_counts.values):
    ax.text(i, v + 1, f'{v}', ha='center', fontsize=9)

# 6. Tipo de empresa
ax = axes[1, 2]
tipo_counts = df_valido['tipo_empresa'].value_counts()
tipo_short = tipo_counts.rename(index={
    '100% uruguaya': 'Uruguaya',
    'Extranjera con personal/operaciones en Uruguay': 'Extranjera (con ops UY)',
    'Uruguaya con capital o control extranjero': 'Uruguaya (capital ext)',
    'Extranjera sin personal/operaciones en Uruguay': 'Extranjera (sin ops UY)'
})
tipo_short = tipo_short.head(4)  # Top 4
tipo_short.plot(kind='bar', ax=ax, color=COLORES_PRINCIPALES[4], edgecolor='white')
ax.set_title('Distribución por Tipo de Empresa', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Frecuencia')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(tipo_short.values):
    ax.text(i, v + 1, f'{v}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('graficos/03_perfil_demografico.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# TABLA CRUZADA: ROL vs OTRAS VARIABLES
# =============================================================================

print("="*70)
print("TABLAS CRUZADAS: ROL vs VARIABLES CLAVE")
print("="*70)

# Rol vs Edad
print("\n--- Rol vs Edad ---")
tabla_rol_edad = pd.crosstab(df_valido['rol'], df_valido['edad'], margins=True, margins_name='Total')
display(tabla_rol_edad)

# Rol vs Antigüedad
print("\n--- Rol vs Antigüedad ---")
tabla_rol_ant = pd.crosstab(df_valido['rol'], df_valido['antiguedad'], margins=True, margins_name='Total')
display(tabla_rol_ant)

# Rol vs Tamaño de empresa
print("\n--- Rol vs Tamaño de Empresa ---")
tabla_rol_tam = pd.crosstab(df_valido['rol'], df_valido['tamanio_empresa_global'], margins=True, margins_name='Total')
display(tabla_rol_tam)

# Rol vs Modalidad
print("\n--- Rol vs Modalidad de Trabajo ---")
tabla_rol_mod = pd.crosstab(df_valido['rol'], df_valido['modalidad_trabajo'], margins=True, margins_name='Total')
display(tabla_rol_mod)

---
## 3. Factores de Retención

In [ ]:
# =============================================================================
# ANÁLISIS DE FACTORES DE RETENCIÓN (Preguntas L a R)
# =============================================================================

print("=" * 70)
print("ANÁLISIS DE FACTORES DE RETENCIÓN")
print("=" * 70)

# Variables de factores de retención
factores_retencion = {
    'oportunidades_desarrollo': 'Oportunidades de desarrollo',
    'feedback_desempenio': 'Feedback sobre desempeño',
    'carga_razonable': 'Carga de trabajo razonable',
    'compensacion_satisfactoria': 'Compensación satisfactoria',
    'cultura_colaborativa': 'Cultura colaborativa',
    'lider_preocupa': 'Líder se preocupa'
}

variable_dependiente = 'buscar_oportunidades'
nombre_dependiente = 'Intención de buscar nuevas oportunidades'

# Verificar que existan las columnas
vars_existentes = [v for v in factores_retencion.keys() if v in df_valido.columns]
print(f"\n Variables de retención encontradas: {len(vars_existentes)}/6")

# --- 1. ESTADÍSTICAS DESCRIPTIVAS POR ROL ---
print("\n" + "=" * 70)
print("1. ESTADÍSTICAS DESCRIPTIVAS POR ROL")
print("=" * 70)

for var, nombre in factores_retencion.items():
    if var in df_valido.columns:
        print(f"\n--- {nombre} ---")
        stats = df_valido.groupby('rol')[var].agg(['count', 'mean', 'std', 'median'])
        stats.columns = ['n', 'Media', 'DE', 'Mediana']
        stats = stats.round(2)
        print(stats)

# Variable dependiente
if variable_dependiente in df_valido.columns:
    print(f"\n--- {nombre_dependiente} ---")
    stats = df_valido.groupby('rol')[variable_dependiente].agg(['count', 'mean', 'std', 'median'])
    stats.columns = ['n', 'Media', 'DE', 'Mediana']
    stats = stats.round(2)
    print(stats)

In [ ]:
# --- 2. VISUALIZACIÓN: FACTORES POR ROL ---
print("\n" + "=" * 70)
print("2. VISUALIZACIÓN: FACTORES DE RETENCIÓN POR ROL")
print("=" * 70)

# Orden fijo de roles para consistencia
orden_roles = ['Colaborador', 'Líder', 'Dirección/Gerencia', 'RRHH']

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

todas_vars = list(factores_retencion.items()) + [(variable_dependiente, nombre_dependiente)]

for i, (var, nombre) in enumerate(todas_vars):
    if var in df_valido.columns and i < len(axes):
        ax = axes[i]
        datos_plot = df_valido[['rol', var]].dropna()
        
        # Calcular medias y ordenar según orden_roles
        medias = datos_plot.groupby('rol')[var].mean()
        medias = medias.reindex(orden_roles)
        
        # Colores fijos según COLORES_ROLES
        colores = [COLORES_ROLES.get(rol, '#999999') for rol in medias.index]
        
        bars = ax.barh(medias.index, medias.values, color=colores, edgecolor='white')
        ax.set_xlabel('Promedio')
        ax.set_title(nombre, fontweight='bold', fontsize=10)
        ax.axvline(x=datos_plot[var].mean(), color='gray', linestyle='--', alpha=0.7)
        
        for bar in bars:
            if not pd.isna(bar.get_width()):
                ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, 
                       f'{bar.get_width():.2f}', va='center', fontsize=9)

# Ocultar subplot extra
if len(todas_vars) < len(axes):
    axes[-1].axis('off')

plt.suptitle('Factores de Retención por Rol', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('graficos/factores_retencion_por_rol.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3. ANÁLISIS DE CORRELACIÓN ---
print("\n" + "=" * 70)
print("3. CORRELACIÓN ENTRE FACTORES Y BÚSQUEDA DE NUEVAS OPORTUNIDADES")
print("=" * 70)

if variable_dependiente in df_valido.columns:
    print(f"\nCorrelación de Spearman con '{nombre_dependiente}':\n")
    
    correlaciones = []
    for var, nombre in factores_retencion.items():
        if var in df_valido.columns:
            from scipy.stats import spearmanr
            datos = df_valido[[var, variable_dependiente]].dropna()
            corr, pval = spearmanr(datos[var], datos[variable_dependiente])
            correlaciones.append({
                'Variable': nombre,
                'Correlación': round(corr, 3),
                'p-valor': round(pval, 4),
                'Significativo': '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
            })
    
    df_corr = pd.DataFrame(correlaciones).sort_values('Correlación')
    print(df_corr.to_string(index=False))
    print("\n* p<0.05, ** p<0.01, *** p<0.001")
    


In [ ]:
# --- 3.1 ANÁLISIS DE CORRELACIÓN POR ROL ---
print("\n" + "=" * 70)
print("3.1 CORRELACIÓN ENTRE FACTORES E INTENCIÓN DE QUEDARSE - POR ROL")
print("=" * 70)

# Usar variable invertida: intencion_quedarse (mayor = más quiere quedarse)
if 'intencion_quedarse' not in df_valido.columns:
    df_valido['intencion_quedarse'] = 7 - pd.to_numeric(df_valido['buscar_oportunidades'], errors='coerce')

variable_retencion = 'intencion_quedarse'  # Variable invertida

from scipy.stats import spearmanr
from IPython.display import display
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

roles = df_valido['rol'].unique()

# Crear DataFrame para almacenar todas las correlaciones
resultados_por_rol = []

for rol in sorted(roles):
    df_rol = df_valido[df_valido['rol'] == rol]
    print(f"\n{'='*50}")
    print(f"ROL: {rol} (n={len(df_rol)})")
    print(f"{'='*50}")
    
    correlaciones_rol = []
    for var, nombre in factores_retencion.items():
        if var in df_rol.columns and variable_retencion in df_rol.columns:
            datos = df_rol[[var, variable_retencion]].dropna()
            if len(datos) > 5:  # Mínimo de datos para correlación
                corr, pval = spearmanr(datos[var], datos[variable_retencion])
                sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
                correlaciones_rol.append({
                    'Variable': nombre,
                    'Correlación': round(corr, 3),
                    'p-valor': round(pval, 4),
                    'Sig': sig
                })
                resultados_por_rol.append({
                    'Rol': rol,
                    'Variable': nombre,
                    'Correlación': round(corr, 3),
                    'p-valor': round(pval, 4)
                })
    
    if correlaciones_rol:
        df_corr_rol = pd.DataFrame(correlaciones_rol).sort_values('Correlación')
        display(df_corr_rol)  # Usar display en vez de print

print("\n* p<0.05, ** p<0.01, *** p<0.001")

# --- Visualización comparativa ---
print("\n" + "=" * 70)
print("COMPARACIÓN VISUAL DE CORRELACIONES POR ROL")
print("=" * 70)

# Crear heatmap de correlaciones por rol
df_pivot = pd.DataFrame(resultados_por_rol).pivot(index='Variable', columns='Rol', values='Correlación')

# Ordenar variables por correlación promedio
df_pivot = df_pivot.loc[df_pivot.mean(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_pivot, annot=True, cmap='RdYlGn', center=0, 
            fmt='.2f', ax=ax, vmin=-1, vmax=1, linewidths=0.5)
ax.set_title('Correlación con Intención de Quedarse\npor Rol (Spearman) - Verde=Positivo', 
             fontweight='bold', fontsize=12)
ax.set_xlabel('Rol')
ax.set_ylabel('Factor de Retención')
plt.tight_layout()
plt.savefig('graficos/correlaciones_retencion_por_rol.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen de hallazgos
print("\n" + "=" * 70)
print("RESUMEN: FACTOR MÁS IMPORTANTE POR ROL")
print("=" * 70)
for rol in df_pivot.columns:
    factor_principal = df_pivot[rol].idxmax()  # El más positivo = más importante
    corr_valor = df_pivot.loc[factor_principal, rol]
    print(f"  • {rol}: {factor_principal} (r = {corr_valor:.2f})")

In [ ]:
# --- 5. HEATMAP DE CORRELACIONES ---
print("\n" + "=" * 70)
print("5. MATRIZ DE CORRELACIONES ENTRE FACTORES")
print("=" * 70)

todas_vars_lista = list(factores_retencion.keys()) + [variable_dependiente]
vars_para_corr = [v for v in todas_vars_lista if v in df_valido.columns]

if len(vars_para_corr) > 1:
    matriz_corr = df_valido[vars_para_corr].corr(method='spearman')
    
    # Renombrar para el gráfico
    nombres_cortos = {
        'oportunidades_desarrollo': 'Oport. Desarrollo',
        'feedback_desempenio': 'Feedback',
        'carga_razonable': 'Carga Razonable',
        'compensacion_satisfactoria': 'Compensación',
        'cultura_colaborativa': 'Cultura',
        'lider_preocupa': 'Líder',
        'buscar_oportunidades': 'Buscar Nuevas Oport.'
    }
    matriz_corr = matriz_corr.rename(index=nombres_cortos, columns=nombres_cortos)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(matriz_corr, annot=True, cmap='RdYlGn', center=0, 
                fmt='.2f', square=True, ax=ax,
                vmin=-1, vmax=1, linewidths=0.5)
    ax.set_title('Correlaciones entre Factores de Retención\n(Spearman)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('graficos/correlaciones_factores_retencion.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 4. Familiaridad y Uso Actual de IA

In [ ]:
# =============================================================================
# ESTADÍSTICAS DESCRIPTIVAS: USO DE IA
# =============================================================================

print("="*70)
print("FAMILIARIDAD Y USO DE IA - ESTADÍSTICAS DESCRIPTIVAS")
print("="*70)

# Variables numéricas de IA (escala 1-6)
vars_ia_numericas = ['familiaridad_ia', 'ia_positiva', 'estimulo_ia_empresa', 'confianza_ia']
vars_ia_existentes = [v for v in vars_ia_numericas if v in df_valido.columns]

if vars_ia_existentes:
    stats_ia = df_valido[vars_ia_existentes].describe().T
    stats_ia['mediana'] = df_valido[vars_ia_existentes].median()
    stats_ia['moda'] = df_valido[vars_ia_existentes].mode().iloc[0]
    
    nombres_descriptivos = {
        'familiaridad_ia': 'Familiaridad con IA',
        'ia_positiva': 'Percepción positiva de IA',
        'estimulo_ia_empresa': 'Estímulo IA en empresa',
        'confianza_ia': 'Confianza en IA'
    }
    stats_ia.index = [nombres_descriptivos.get(i, i) for i in stats_ia.index]
    
    print("\nEstadísticas (escala 1-6):")
    print(stats_ia[['count', 'mean', 'std', 'min', 'mediana', 'max']].round(2).to_string())

In [ ]:
# =============================================================================
# FRECUENCIA DE USO DE IA
# =============================================================================

print("\n--- Frecuencia de Uso de IA ---")
if 'frecuencia_uso_ia' in df_valido.columns:
    # Limpiar valores inconsistentes
    df_valido['frecuencia_uso_ia_clean'] = df_valido['frecuencia_uso_ia'].replace({
        '5': 'A diario',
        '6': 'A diario',
        '4': 'Varias veces por semana'
    })
    
    freq_uso = df_valido['frecuencia_uso_ia_clean'].value_counts()
    for val, count in freq_uso.items():
        print(f"  {val}: {count} ({count/len(df_valido)*100:.1f}%)")

In [ ]:
# =============================================================================
# GRÁFICO: FAMILIARIDAD Y PERCEPCIÓN DE IA
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Familiaridad con IA
ax = axes[0, 0]
if 'familiaridad_ia' in df_valido.columns:
    fam_counts = df_valido['familiaridad_ia'].value_counts().sort_index()
    bars = ax.bar(fam_counts.index.astype(int), fam_counts.values, 
                  color=COLORES_LIKERT[::1][:len(fam_counts)], edgecolor='white')
    ax.set_xlabel('Nivel de familiaridad (1=Nada, 6=Muy familiarizado)')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Familiaridad con el Concepto de IA', fontweight='bold')
    ax.set_xticks(range(1, 7))
    for bar, val in zip(bars, fam_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val}', ha='center', fontsize=9)

# 2. Percepción positiva de IA
ax = axes[0, 1]
if 'ia_positiva' in df_valido.columns:
    pos_counts = df_valido['ia_positiva'].value_counts().sort_index()
    bars = ax.bar(pos_counts.index.astype(int), pos_counts.values, 
                  color=COLORES_LIKERT[::1][:len(pos_counts)], edgecolor='white')
    ax.set_xlabel('Nivel de acuerdo (1=Muy negativo, 6=Muy positivo)')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Percepción de IA como Algo Positivo', fontweight='bold')
    ax.set_xticks(range(1, 7))
    for bar, val in zip(bars, pos_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val}', ha='center', fontsize=9)

# 3. Frecuencia de uso
ax = axes[1, 0]
if 'frecuencia_uso_ia_clean' in df_valido.columns:
    orden_freq = ['A diario', 'Varias veces por semana', 'Aproximadamente una vez por semana',
                  'Varias veces al mes', 'Rara vez (menos de una vez al mes)']
    freq_counts = df_valido['frecuencia_uso_ia_clean'].value_counts().reindex(orden_freq).dropna()
    bars = ax.barh(range(len(freq_counts)), freq_counts.values, color=COLORES_PRINCIPALES[0])
    ax.set_yticks(range(len(freq_counts)))
    ax.set_yticklabels(freq_counts.index)
    ax.set_xlabel('Frecuencia')
    ax.set_title('Frecuencia de Uso de Herramientas de IA', fontweight='bold')
    ax.invert_yaxis()
    for bar, val in zip(bars, freq_counts.values):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val} ({val/len(df_valido)*100:.0f}%)', 
                va='center', fontsize=9)

# 4. Estímulo de IA en la empresa
ax = axes[1, 1]
if 'estimulo_ia_empresa' in df_valido.columns:
    est_counts = df_valido['estimulo_ia_empresa'].value_counts().sort_index()
    bars = ax.bar(est_counts.index.astype(int), est_counts.values, 
                  color=COLORES_LIKERT[::1][:len(est_counts)], edgecolor='white')
    ax.set_xlabel('Nivel de estímulo (1=Nada, 6=Mucho)')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Estímulo del Uso de IA en la Empresa', fontweight='bold')
    ax.set_xticks(range(1, 7))
    for bar, val in zip(bars, est_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('graficos/04_uso_ia.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# COMPARACIÓN: USO DE IA POR ROL
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

vars_comparar = ['familiaridad_ia', 'ia_positiva', 'confianza_ia']
titulos = ['Familiaridad con IA', 'Percepción Positiva de IA', 'Confianza en IA']

for ax, var, titulo in zip(axes, vars_comparar, titulos):
    if var in df_valido.columns:
        datos_por_rol = df_valido.groupby('rol')[var].mean().sort_values(ascending=True)
        colors = [COLORES_ROLES.get(r, '#888888') for r in datos_por_rol.index]
        bars = ax.barh(datos_por_rol.index, datos_por_rol.values, color=colors)
        ax.set_xlabel('Promedio (escala 1-6)')
        ax.set_title(titulo, fontweight='bold')
        ax.set_xlim(0, 6)
        ax.axvline(x=df_valido[var].mean(), color='gray', linestyle='--', alpha=0.7, label='Promedio general')
        
        for bar, val in zip(bars, datos_por_rol.values):
            ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('graficos/05_uso_ia_por_rol.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Percepción por Rol: Gestión de Talento con IA

In [ ]:
# =============================================================================
# DEFINIR COLUMNAS POR ROL
# =============================================================================
# Estructura del cuestionario:
# - Colaboradores (93): columnas 26-35 (col_*)
# - Líderes (50): columnas 36-45 (lid_*)
# - Dirección/Gerencia (29): columnas 46-55 (rrhh_*)
# - RRHH (21): columnas 56-65 (rrhh_*)
# - Generales (todos): columnas 66+ (ia_invasivo, cultura_no_preparada, impulsaria_ia)

# === COLUMNAS COLABORADORES ===
cols_colaboradores = [
    'col_ia_detectar_desmotivacion',
    'col_ia_capacitacion', 
    'col_ia_carrera',
    'col_ia_equilibrio_personal',
    'col_ia_apoyo_emocional',
    'col_ia_interacciones',
    'col_ia_asistente_admin',
    'col_ia_emociones',
    'col_ia_asistente_personal',
    'col_ia_transparencia'
]

# === COLUMNAS LÍDERES ===
cols_lideres = [
    'lid_ia_detectar_desmotivacion',
    'lid_ia_capacitacion',
    'lid_ia_movilidad',
    'lid_ia_alertas_sobrecarga',
    'lid_ia_comunicacion',
    'lid_ia_agente_admin',
    'lid_ia_expresiones_preocupacion',
    'lid_incomodidad_decisiones_auto',
    'lid_equipo_comodo_asistente',
    'lid_disposicion_con_etica'
    
]

# === COLUMNAS DIRECCIÓN/GERENCIA ===
cols_direccion = [
    'dir_ger_ia_fuga_talento',
    'dir_ger_ia_formacion_personalizada',
    'dir_ger_ia_sucesion',
    'dir_ger_ia_deteccion_agotamiento',
    'dir_ger_ia_patrones_comunicacion',
    'dir_ger_ia_asistente_consultas',
    'dir_ger_ia_interpretar_emociones',
    'dir_ger_ia_compensaciones',
    'dir_ger_preocupa_vigilancia',
    'dir_ger_valor_con_politicas'
]

# === COLUMNAS RRHH ===
cols_rrhh = [
    'rrhh_ia_historicos_retencion',
    'rrhh_ia_brechas_habilidades',
    'rrhh_ia_movimientos_internos',
    'rrhh_ia_desgaste_emocional',
    'rrhh_ia_analisis_lenguaje',
    'rrhh_ia_redes_colaboracion',
    'rrhh_ia_asistente_eficiencia',
    'rrhh_ia_recomendaciones_salario',
    'rrhh_desconfianza_vigilancia',
    'rrhh_politicas_claras'
]

# === COLUMNAS GENERALES (todos responden) ===
cols_generales = [
    'ia_invasivo',
    'cultura_no_preparada',
    'impulsaria_ia'
]

# === NOMBRES DESCRIPTIVOS ===
nombres_cols = {
    # Colaboradores
    'col_ia_detectar_desmotivacion': 'Detectar desmotivación\n(tareas, reuniones)',
    'col_ia_capacitacion': 'Recomendaciones de\ncapacitación',
    'col_ia_carrera': 'Sugerencias de\ncambios de rol',
    'col_ia_equilibrio_personal': 'Equilibrio personal/laboral',
    'col_ia_apoyo_emocional': 'Apoyo emocional\n(bienestar)',
    'col_ia_interacciones': 'Análisis de\ninteracciones',
    'col_ia_asistente_admin': 'Asistente virtual\nadministrativo',
    'col_ia_emociones': 'Análisis de\nemociones',
    'col_ia_asistente_personal': 'Asistente virtual\npersonal',
    'col_ia_transparencia': 'Solo con\ntransparencia',
    
    # Líderes
    'lid_ia_detectar_desmotivacion': 'Detectar desmotivación\nen equipo',
    'lid_ia_capacitacion': 'Capacitación para\nequipo',
    'lid_ia_movilidad': 'Movimientos internos',
    'lid_ia_alertas_sobrecarga': 'Alertas de\nsobrecarga',
    'lid_ia_comunicacion': 'Monitoreo de\ncomunicación',
    'lid_ia_agente_admin': 'Agente virtual\npara equipo',
    'lid_incomodidad_decisiones_auto': 'Incomodidad con\ndecisiones auto',
    'lid_equipo_comodo_asistente': 'Equipo cómodo con\nasistente',
    'lid_disposicion_con_etica': 'Disposición con\nética',
    'lid_ia_expresiones_preocupacion': 'Expresiones genera\npreocupación',
    
    # Dirección/Gerencia
    'dir_ger_ia_fuga_talento': 'Anticipar fuga\nde talento',
    'dir_ger_ia_formacion_personalizada': 'Formación\npersonalizada',
    'dir_ger_ia_sucesion': 'Sugerencias de\nsucesión',
    'dir_ger_ia_deteccion_agotamiento': 'Detectar\nagotamiento',
    'dir_ger_ia_patrones_comunicacion': 'Patrones de\ncomunicación',
    'dir_ger_ia_asistente_consultas': 'Asistente para\nconsultas',
    'dir_ger_ia_interpretar_emociones': 'Interpretar\nemociones',
    'dir_ger_ia_compensaciones': 'Recomendaciones\nsalariales',
    'dir_ger_preocupa_vigilancia': 'Preocupa percepción\nde vigilancia',
    'dir_ger_valor_con_politicas': 'Valor con políticas\nclaras',
    
    # RRHH
    'rrhh_ia_historicos_retencion': 'Análisis históricos\nretención',
    'rrhh_ia_brechas_habilidades': 'Brechas de\nhabilidades',
    'rrhh_ia_movimientos_internos': 'Movimientos\ninternos',
    'rrhh_ia_desgaste_emocional': 'Desgaste\nemocional',
    'rrhh_ia_analisis_lenguaje': 'Análisis de\nlenguaje',
    'rrhh_ia_redes_colaboracion': 'Redes de\ncolaboración',
    'rrhh_ia_asistente_eficiencia': 'Asistente para\neficiencia',
    'rrhh_ia_recomendaciones_salario': 'Recomendaciones\nsalariales',
    'rrhh_desconfianza_vigilancia': 'Desconfianza por\nvigilancia',
    'rrhh_politicas_claras': 'Políticas claras\nnecesarias',
    
    # Generales
    'ia_invasivo': 'IA podría ser\ninvasivo',
    'cultura_no_preparada': 'Cultura no\npreparada',
    'impulsaria_ia': 'Impulsaría IA\nen empresa'
}

print(" Columnas definidas por rol:")
print(f"   - Colaboradores: {len(cols_colaboradores)} preguntas (n=93)")
print(f"   - Líderes: {len(cols_lideres)} preguntas (n=50)")
print(f"   - Dirección/Gerencia: {len(cols_direccion)} preguntas (n=29)")
print(f"   - RRHH: {len(cols_rrhh)} preguntas (n=21)")
print(f"   - Generales: {len(cols_generales)} preguntas (n=193)")

In [ ]:
# =============================================================================
# FUNCIÓN PARA CREAR GRÁFICO DE BARRAS DIVERGENTES (LIKERT)
# =============================================================================

def grafico_likert_divergente(df, columnas, nombres, titulo, filename):
    """
    Crea un gráfico de barras divergentes para escalas Likert
    Muestra respuestas negativas (1-3) a la izquierda y positivas (4-6) a la derecha
    """
    # Filtrar columnas que existen
    cols_existentes = [c for c in columnas if c in df.columns]
    
    if not cols_existentes:
        print(f"No hay datos para: {titulo}")
        return None
    
    # Calcular porcentajes por categoría
    resultados = []
    for col in cols_existentes:
        datos = df[col].dropna()
        total = len(datos)
        if total == 0:
            continue
        
        pcts = {}
        for i in range(1, 7):
            pcts[i] = (datos == i).sum() / total * 100
        
        pcts['columna'] = col
        pcts['nombre'] = nombres.get(col, col)
        pcts['n'] = total
        pcts['media'] = datos.mean()
        resultados.append(pcts)
    
    if not resultados:
        print(f"No hay datos válidos para: {titulo}")
        return None
    
    df_plot = pd.DataFrame(resultados)
    
    # Crear figura
    fig, ax = plt.subplots(figsize=(12, max(4, len(df_plot) * 0.8)))
    
    y_pos = range(len(df_plot))
    
    # Colores para cada nivel (1=rojo oscuro, 6=azul oscuro)
    colores = COLORES_LIKERT
    
    # Dibujar barras negativas (1, 2, 3) hacia la izquierda
    left_1 = -df_plot[1] - df_plot[2] - df_plot[3]
    left_2 = -df_plot[2] - df_plot[3]
    left_3 = -df_plot[3]
    
    ax.barh(y_pos, -df_plot[1], left=left_1 + df_plot[1], color=colores[0], label='1 - Muy en desacuerdo', height=0.6)
    ax.barh(y_pos, -df_plot[2], left=left_2 + df_plot[2], color=colores[1], label='2', height=0.6)
    ax.barh(y_pos, -df_plot[3], left=left_3 + df_plot[3], color=colores[2], label='3', height=0.6)
    
    # Dibujar barras positivas (4, 5, 6) hacia la derecha
    ax.barh(y_pos, df_plot[4], left=0, color=colores[3], label='4', height=0.6)
    ax.barh(y_pos, df_plot[5], left=df_plot[4], color=colores[4], label='5', height=0.6)
    ax.barh(y_pos, df_plot[6], left=df_plot[4] + df_plot[5], color=colores[5], label='6 - Muy de acuerdo', height=0.6)
    
    # Configurar ejes
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{row['nombre']}\n(n={row['n']}, μ={row['media']:.1f})" for _, row in df_plot.iterrows()])
    ax.set_xlabel('Porcentaje de respuestas')
    ax.set_title(titulo, fontweight='bold', fontsize=14)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlim(-100, 100)
    
    # Añadir etiquetas de porcentaje
    for i, row in df_plot.iterrows():
        # Porcentaje negativo (1-3)
        neg_total = row[1] + row[2] + row[3]
        if neg_total > 5:
            ax.text(-neg_total/2, i, f'{neg_total:.0f}%', ha='center', va='center', fontsize=8, color='black', fontweight='bold')
        
        # Porcentaje positivo (4-6)
        pos_total = row[4] + row[5] + row[6]
        if pos_total > 5:
            ax.text(pos_total/2, i, f'{pos_total:.0f}%', ha='center', va='center', fontsize=8, color='black', fontweight='bold')
    
    # Leyenda
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=6, fontsize=8)
    
    plt.tight_layout()
    plt.savefig(f'graficos/{filename}', dpi=150, bbox_inches='tight')
    plt.show()
    
    return df_plot

In [ ]:
# =============================================================================
# DIAGNÓSTICO: VERIFICAR CÁLCULOS DE COLABORADORES
# =============================================================================

print("=" * 70)
print("VERIFICACIÓN DE DATOS - COLABORADORES")
print("=" * 70)

df_colaboradores = df_valido[df_valido['rol'] == 'Colaborador'].copy()
print(f"\nTotal colaboradores: {len(df_colaboradores)}")

# Verificar cada columna
print("\n" + "=" * 70)
print("DETALLE POR VARIABLE:")
print("=" * 70)

for col in cols_colaboradores:
    if col in df_colaboradores.columns:
        datos = df_colaboradores[col].dropna()
        print(f"\n--- {nombres_cols.get(col, col)} ---")
        print(f"Columna: {col}")
        print(f"n válidos: {len(datos)} de {len(df_colaboradores)}")
        print(f"Valores únicos: {sorted(datos.unique())}")
        print(f"\nConteo por valor:")
        conteo = datos.value_counts().sort_index()
        for val, count in conteo.items():
            pct = count / len(datos) * 100
            print(f"  {val}: {count} ({pct:.1f}%)")
        
        # Calcular positivos vs negativos
        neg = ((datos >= 1) & (datos <= 3)).sum()
        pos = ((datos >= 4) & (datos <= 6)).sum()
        print(f"\nResumen:")
        print(f"  Negativos (1-3): {neg} ({neg/len(datos)*100:.1f}%)")
        print(f"  Positivos (4-6): {pos} ({pos/len(datos)*100:.1f}%)")
        print(f"  Media: {datos.mean():.2f}")

In [ ]:
# =============================================================================
# GRÁFICO: PERCEPCIÓN COLABORADORES (n=93)
# =============================================================================

# Filtrar solo colaboradores
df_colaboradores = df_valido[df_valido['rol'] == 'Colaborador'].copy()

# Seleccionar columnas que existen
cols_graf_col = [c for c in cols_colaboradores if c in df_colaboradores.columns][:10]

if cols_graf_col and len(df_colaboradores) > 0:
    print(f"Analizando {len(df_colaboradores)} colaboradores")
    df_result = grafico_likert_divergente(
        df_colaboradores, 
        cols_graf_col,
        nombres_cols,
        f'Percepción de Colaboradores (n={len(df_colaboradores)})',
        '06_percepcion_colaboradores.png'
    )
else:
    print(" No hay datos de colaboradores para graficar")

In [ ]:
# =============================================================================
# GRÁFICO: PERCEPCIÓN LÍDERES (n=50)
# =============================================================================

# Filtrar solo líderes
df_lideres = df_valido[df_valido['rol'] == 'Líder'].copy()

# Seleccionar columnas que existen
cols_graf_lid = [c for c in cols_lideres if c in df_lideres.columns][:10]

if cols_graf_lid and len(df_lideres) > 0:
    print(f"Analizando {len(df_lideres)} líderes")
    df_result = grafico_likert_divergente(
        df_lideres, 
        cols_graf_lid,
        nombres_cols,
        f'Percepción de Líderes (n={len(df_lideres)})',
        '07_percepcion_lideres.png'
    )
else:
    print("No hay datos de líderes para graficar")

In [ ]:
# =============================================================================
# GRÁFICO: PERCEPCIÓN DIRECCIÓN/GERENCIA (n=29) Y RRHH (n=21)
# =============================================================================

# --- DIRECCIÓN/GERENCIA ---
df_direccion = df_valido[df_valido['rol'] == 'Dirección/Gerencia'].copy()
cols_graf_dir = [c for c in cols_direccion if c in df_direccion.columns][:10]

if cols_graf_dir and len(df_direccion) > 0:
    print(f"Analizando {len(df_direccion)} de Dirección/Gerencia")
    df_result = grafico_likert_divergente(
        df_direccion, 
        cols_graf_dir,
        nombres_cols,
        f'Percepción de Dirección/Gerencia (n={len(df_direccion)})',
        '08_percepcion_direccion.png'
    )

# --- RRHH ---
df_rrhh = df_valido[df_valido['rol'] == 'RRHH'].copy()
cols_graf_rrhh = [c for c in cols_rrhh if c in df_rrhh.columns][:10]

if cols_graf_rrhh and len(df_rrhh) > 0:
    print(f"Analizando {len(df_rrhh)} de RRHH")
    df_result = grafico_likert_divergente(
        df_rrhh, 
        cols_graf_rrhh,
        nombres_cols,
        f'Percepción de RRHH (n={len(df_rrhh)})',
        '09_percepcion_rrhh.png'
    )
else:
    print("No hay datos de RRHH para graficar")

---
## 6. Percepción por Rol: Comunicación y Emociones

In [ ]:
# =============================================================================
# DEFINIR COLUMNAS DE COMUNICACIÓN/EMOCIONES POR ROL
# =============================================================================

# Colaboradores - comunicación y emociones
cols_com_colaboradores = [
    'col_ia_interacciones',
    'col_ia_emociones',
    'col_ia_asistente_personal',
    'col_ia_asistente_admin'
]

# Líderes - comunicación y emociones
cols_com_lideres = [
    'lid_ia_comunicacion',
    'lid_ia_agente_admin',
    'lid_equipo_comodo_asistente',
    'lid_ia_expresiones_preocupacion'
]

# Dirección/Gerencia - comunicación y emociones
cols_com_direccion = [
    'dir_ger_ia_patrones_comunicacion',
    'dir_ger_ia_interpretar_emociones',
    'dir_ger_ia_asistente_consultas'
]

# RRHH - comunicación y emociones
cols_com_rrhh = [
    'rrhh_ia_analisis_lenguaje',
    'rrhh_ia_redes_colaboracion',
    'rrhh_ia_asistente_eficiencia'
]

print("Columnas de comunicación/emociones definidas por rol")

In [ ]:
# =============================================================================
# GRÁFICO: COMUNICACIÓN/EMOCIONES - COLABORADORES
# =============================================================================

df_colaboradores = df_valido[df_valido['rol'] == 'Colaborador'].copy()
cols_graf = [c for c in cols_com_colaboradores if c in df_colaboradores.columns]

if cols_graf and len(df_colaboradores) > 0:
    df_result = grafico_likert_divergente(
        df_colaboradores, 
        cols_graf,
        nombres_cols,
        f'Colaboradores: Comunicación y Emociones (n={len(df_colaboradores)})',
        '10_comunicacion_colaboradores.png'
    )

In [ ]:
# =============================================================================
# GRÁFICO: COMUNICACIÓN/EMOCIONES - LÍDERES
# =============================================================================

df_lideres = df_valido[df_valido['rol'] == 'Líder'].copy()
cols_graf = [c for c in cols_com_lideres if c in df_lideres.columns]

if cols_graf and len(df_lideres) > 0:
    df_result = grafico_likert_divergente(
        df_lideres, 
        cols_graf,
        nombres_cols,
        f'Líderes: Comunicación y Emociones (n={len(df_lideres)})',
        '11_comunicacion_lideres.png'
    )

In [ ]:
# =============================================================================
# GRÁFICO: COMUNICACIÓN/EMOCIONES - DIRECCIÓN/GERENCIA Y RRHH
# =============================================================================

# Dirección/Gerencia
df_direccion = df_valido[df_valido['rol'] == 'Dirección/Gerencia'].copy()
cols_graf = [c for c in cols_com_direccion if c in df_direccion.columns]

if cols_graf and len(df_direccion) > 0:
    df_result = grafico_likert_divergente(
        df_direccion, 
        cols_graf,
        nombres_cols,
        f'Dirección/Gerencia: Comunicación (n={len(df_direccion)})',
        '12_comunicacion_direccion.png'
    )

# RRHH
df_rrhh = df_valido[df_valido['rol'] == 'RRHH'].copy()
cols_graf = [c for c in cols_com_rrhh if c in df_rrhh.columns]

if cols_graf and len(df_rrhh) > 0:
    df_result = grafico_likert_divergente(
        df_rrhh, 
        cols_graf,
        nombres_cols,
        f'RRHH: Comunicación (n={len(df_rrhh)})',
        '13_comunicacion_rrhh.png'
    )

---
## 7. Preocupaciones Éticas y de Privacidad

In [ ]:
# =============================================================================
# COLUMNAS DE ÉTICA Y PRIVACIDAD POR ROL
# =============================================================================

# Colaboradores - ética
cols_etica_colaboradores = [
    'col_ia_transparencia',
    'col_ia_emociones',
    'col_ia_interacciones']

# Líderes - ética
cols_etica_lideres = [
    'lid_incomodidad_decisiones_auto',
    'lid_disposicion_con_etica',
    'lid_ia_expresiones_preocupacion'
]

# Dirección/Gerencia - ética
cols_etica_direccion = [
    'dir_ger_preocupa_vigilancia',
    'dir_ger_valor_con_politicas',
    'dir_ger_ia_interpretar_emociones'
]

# RRHH - ética
cols_etica_rrhh = [
    'rrhh_desconfianza_vigilancia',
    'rrhh_politicas_claras',
    'rrhh_ia_analisis_lenguaje'
]

# Columnas generales NUMÉRICAS (TODOS responden) - escala 1-6
cols_etica_general = [
    'ia_invasivo',           # Col 70: IA podría ser invasivo (1-6)
    'cultura_no_preparada'   # Col 71: Cultura no preparada (1-6)
]
# Nota: 'impulsaria_ia' (Col 72) es Si/No, no numérica

print("Columnas de ética/privacidad definidas por rol")
print("Nota: impulsaria_ia es Si/No, se analiza por separado")

In [ ]:
# =============================================================================
# ANÁLISIS ÉTICO: DOS DIMENSIONES
# =============================================================================

print("=" * 70)
print("ANÁLISIS ÉTICO: PREOCUPACIÓN vs VALORACIÓN DE GOBERNANZA")
print("=" * 70)

# -----------------------------------------------------------------------------
# DIMENSIÓN 1: Preocupación por Privacidad/Control
# -----------------------------------------------------------------------------
# Mide qué tan preocupados están por el uso invasivo/controlador de la IA

# Variables que miden PREOCUPACIÓN directamente (alto = más preocupado)
vars_preocupacion_directa = {
    'Colaborador': [],  # Las de colaborador son de aceptación, van en invertir
    'Líder': ['lid_incomodidad_decisiones_auto', 'lid_ia_expresiones_preocupacion'],
    'Dirección/Gerencia': ['dir_ger_preocupa_vigilancia'],
    'RRHH': ['rrhh_desconfianza_vigilancia']
}

# Variables que miden ACEPTACIÓN (bajo = más preocupado) - SE INVIERTEN
vars_preocupacion_invertir = {
    'Colaborador': ['col_ia_emociones', 'col_ia_interacciones'],
    'Líder': [],
    'Dirección/Gerencia': ['dir_ger_ia_interpretar_emociones'],
    'RRHH': ['rrhh_ia_analisis_lenguaje']
}

# Variable general (todos responden) - mide preocupación directamente
var_general_preocupacion = 'ia_invasivo'

# -----------------------------------------------------------------------------
# DIMENSIÓN 2: Valoración de Transparencia y Políticas
# -----------------------------------------------------------------------------
# Mide qué tanto valoran la transparencia y políticas claras

vars_valoracion = {
    'Colaborador': ['col_ia_transparencia'],
    'Líder': ['lid_disposicion_con_etica'],
    'Dirección/Gerencia': ['dir_ger_valor_con_politicas'],
    'RRHH': ['rrhh_politicas_claras']
}

# -----------------------------------------------------------------------------
# CALCULAR SCORES POR ROL
# -----------------------------------------------------------------------------

resultados = []
orden_roles = ['Colaborador', 'Líder', 'Dirección/Gerencia', 'RRHH']

print("\nVariables utilizadas:")
print("\n  PREOCUPACIÓN (directa - alto = más preocupado):")
print("    - ia_invasivo (todos)")
print("    - lid_incomodidad_decisiones_auto (Líder)")
print("    - lid_ia_expresiones_preocupacion (Líder)")
print("    - dir_ger_preocupa_vigilancia (Dirección)")
print("    - rrhh_desconfianza_vigilancia (RRHH)")
print("\n  PREOCUPACIÓN (invertida - bajo original = más preocupado):")
print("    - col_ia_emociones (Colaborador) -> 7-valor")
print("    - col_ia_interacciones (Colaborador) -> 7-valor")
print("    - dir_ger_ia_interpretar_emociones (Dirección) -> 7-valor")
print("    - rrhh_ia_analisis_lenguaje (RRHH) -> 7-valor")
print("\n  VALORACIÓN (directa - alto = más valora):")
print("    - col_ia_transparencia (Colaborador)")
print("    - lid_disposicion_con_etica (Líder)")
print("    - dir_ger_valor_con_politicas (Dirección)")
print("    - rrhh_politicas_claras (RRHH)")

for rol in orden_roles:
    df_rol = df_valido[df_valido['rol'] == rol].copy()
    
    if len(df_rol) == 0:
        continue
    
    # Score de Preocupación
    preocupacion_valores = []
    
    # Variable general (directa)
    if var_general_preocupacion in df_rol.columns:
        val = pd.to_numeric(df_rol[var_general_preocupacion], errors='coerce').mean()
        if not pd.isna(val):
            preocupacion_valores.append(val)
    
    # Variables específicas DIRECTAS (alto = más preocupado)
    for var in vars_preocupacion_directa.get(rol, []):
        if var in df_rol.columns:
            val = pd.to_numeric(df_rol[var], errors='coerce').mean()
            if not pd.isna(val):
                preocupacion_valores.append(val)
    
    # Variables específicas A INVERTIR (bajo = más preocupado -> invertir)
    for var in vars_preocupacion_invertir.get(rol, []):
        if var in df_rol.columns:
            val = pd.to_numeric(df_rol[var], errors='coerce').mean()
            if not pd.isna(val):
                preocupacion_valores.append(7 - val)  # INVERTIR: 7 - valor
    
    score_preocupacion = np.mean(preocupacion_valores) if preocupacion_valores else np.nan
    
    # Score de Valoración (todas directas)
    valoracion_valores = []
    for var in vars_valoracion.get(rol, []):
        if var in df_rol.columns:
            val = pd.to_numeric(df_rol[var], errors='coerce').mean()
            if not pd.isna(val):
                valoracion_valores.append(val)
    
    score_valoracion = np.mean(valoracion_valores) if valoracion_valores else np.nan
    
    resultados.append({
        'Rol': rol,
        'Preocupación Privacidad/Control': score_preocupacion,
        'Valoración Transparencia/Políticas': score_valoracion,
        'n': len(df_rol)
    })

df_resultados = pd.DataFrame(resultados)

print("\n" + "=" * 70)
print("SCORES POR DIMENSIÓN ÉTICA (escala 1-6)")
print("=" * 70)
print("  • Preocupación: Mayor valor = Más preocupados por privacidad/control")
print("  • Valoración: Mayor valor = Más valoran transparencia y políticas")
print()
display(df_resultados.round(2))

# -----------------------------------------------------------------------------
# GRÁFICO COMPARATIVO
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Preocupación por Privacidad/Control
ax1 = axes[0]
df_plot1 = df_resultados.sort_values('Preocupación Privacidad/Control', ascending=True)
colores1 = [COLORES_ROLES.get(rol, '#888888') for rol in df_plot1['Rol']]
bars1 = ax1.barh(df_plot1['Rol'], df_plot1['Preocupación Privacidad/Control'], 
                  color=colores1, edgecolor='white', height=0.6)
ax1.axvline(x=3.5, color='gray', linestyle='--', alpha=0.7)
ax1.set_xlim(1, 6)
ax1.set_xlabel('Score (1-6)', fontsize=10)
ax1.set_title('Preocupación por Privacidad/Control\n(Mayor = Más preocupados)', fontweight='bold', fontsize=12)

for i, (_, row) in enumerate(df_plot1.iterrows()):
    if not pd.isna(row['Preocupación Privacidad/Control']):
        ax1.text(row['Preocupación Privacidad/Control'] + 0.1, i, 
                f"{row['Preocupación Privacidad/Control']:.2f}", va='center', fontsize=11)

# Gráfico 2: Valoración de Transparencia y Políticas
ax2 = axes[1]
df_plot2 = df_resultados.sort_values('Valoración Transparencia/Políticas', ascending=True)
colores2 = [COLORES_ROLES.get(rol, '#888888') for rol in df_plot2['Rol']]
bars2 = ax2.barh(df_plot2['Rol'], df_plot2['Valoración Transparencia/Políticas'], 
                  color=colores2, edgecolor='white', height=0.6)
ax2.axvline(x=3.5, color='gray', linestyle='--', alpha=0.7)
ax2.set_xlim(1, 6)
ax2.set_xlabel('Score (1-6)', fontsize=10)
ax2.set_title('Valoración de Transparencia y Políticas\n(Mayor = Más valoran)', fontweight='bold', fontsize=12)

for i, (_, row) in enumerate(df_plot2.iterrows()):
    if not pd.isna(row['Valoración Transparencia/Políticas']):
        ax2.text(row['Valoración Transparencia/Políticas'] + 0.1, i, 
                f"{row['Valoración Transparencia/Políticas']:.2f}", va='center', fontsize=11)

plt.suptitle('Dimensiones Éticas en el Uso de IA por Rol', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('graficos/07_dimensiones_eticas_por_rol.png', dpi=150, bbox_inches='tight')
plt.show()

# -----------------------------------------------------------------------------
# ANÁLISIS: ¿Los más preocupados valoran más la gobernanza?
# -----------------------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRELACIÓN: ¿Los más preocupados valoran más la gobernanza?")
print("=" * 70)

# Correlación entre ambas dimensiones a nivel de rol
if len(df_resultados) >= 3:
    corr_dims = df_resultados[['Preocupación Privacidad/Control', 'Valoración Transparencia/Políticas']].corr().iloc[0,1]
    print(f"\nCorrelación entre dimensiones (a nivel de rol): {corr_dims:.3f}")
    if corr_dims > 0.5:
        print("-> Los roles más preocupados también valoran más la transparencia y políticas")
    elif corr_dims < -0.5:
        print("-> Relación inversa: los más preocupados valoran menos la gobernanza")
    else:
        print("-> No hay relación fuerte entre preocupación y valoración de gobernanza a nivel de rol")

In [ ]:
# =============================================================================
# GRÁFICO: PREOCUPACIONES ÉTICAS - TODOS LOS ROLES
# =============================================================================

# Usar columnas generales numéricas (escala 1-6)
cols_graf = [c for c in cols_etica_general if c in df_valido.columns]

if cols_graf:
    print(f"Analizando preocupaciones éticas (n={len(df_valido)})")
    
    # Orden fijo de roles
    orden_roles = ['Colaborador', 'Líder', 'Dirección/Gerencia', 'RRHH']
    
    # Crear gráfico comparativo por rol
    fig, axes = plt.subplots(1, len(cols_graf), figsize=(6*len(cols_graf), 6))
    if len(cols_graf) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, cols_graf):
        # Convertir a numérico por si acaso
        df_valido[col] = pd.to_numeric(df_valido[col], errors='coerce')
        datos_por_rol = df_valido.groupby('rol')[col].mean().reindex(orden_roles)
        
        # Colores fijos por rol usando COLORES_ROLES
        colores = [COLORES_ROLES.get(rol, '#999999') for rol in datos_por_rol.index]
        
        bars = ax.barh(datos_por_rol.index, datos_por_rol.values, color=colores, edgecolor='white')
        
        ax.set_xlabel('Promedio (1-6)')
        ax.set_title(nombres_cols.get(col, col).replace('\\n', ' '), fontweight='bold')
        ax.set_xlim(1, 6)
        ax.axvline(x=3.5, color='gray', linestyle='--', alpha=0.5)
        
        # Agregar valores
        for bar in bars:
            width = bar.get_width()
            if not pd.isna(width):
                ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, 
                       f'{width:.2f}', va='center', fontsize=10)
    
    plt.suptitle('Preocupaciones Éticas por Rol', fontweight='bold', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('graficos/14_etica_por_rol.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# GRÁFICO: DISPOSICIÓN A IMPULSAR IA POR ROL (Si/No)
# =============================================================================

if 'impulsaria_ia' in df_valido.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Calcular porcentaje de "Si" por rol
    tabla = pd.crosstab(df_valido['rol'], df_valido['impulsaria_ia'], normalize='index') * 100
    
    if 'Si' in tabla.columns:
        pct_si = tabla['Si'].sort_values(ascending=True)
        
        colors = [COLORES_ROLES.get(r, '#888888') for r in pct_si.index]
        
        bars = ax.barh(pct_si.index, pct_si.values, color=colors)
        
        ax.set_xlabel('Porcentaje que impulsaría IA (%)')
        ax.set_title('Disposición a Impulsar IA por Rol', fontweight='bold', fontsize=14)
        ax.set_xlim(0, 100)
        ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5)
        
        # Agregar valores
        for bar, val in zip(bars, pct_si.values):
            n_rol = (df_valido['rol'] == bar.get_y()).sum()
            ax.text(val + 2, bar.get_y() + bar.get_height()/2, 
                   f'{val:.1f}%', va='center', fontsize=11)
        
        # Mostrar totales
        total_si = (df_valido['impulsaria_ia'] == 'Si').sum()
        total = len(df_valido)
        ax.set_title(f'Disposición a Impulsar IA por Rol\n(Total: {total_si}/{total} = {total_si/total*100:.1f}% respondieron "Sí")', 
                    fontweight='bold', fontsize=12)
        
        plt.tight_layout()
        plt.savefig('graficos/15_impulsar_ia_por_rol.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("No hay respuestas 'Si' en impulsaria_ia")

---
## 8. Análisis por Tecnología de IA

Análisis de la percepción de diferentes tecnologías de IA aplicadas a RRHH:

| Código | Tecnología |
|--------|------------|
| **PA** | People Analytics integrales |
| **UP/R** | IA para upskilling/reskilling |
| **CHAT** | Chatbots y asistentes virtuales de RRHH |
| **ESA** | Employee Sentiment Analysis (emociones, voz, texto) |
| **WELL** | Plataformas de bienestar emocional basadas en IA |
| **MOB** | Motores de recomendación para movilidad interna |
| **ONA** | Organizational Network Analysis |
| **COMP** | Sistemas de compensación personalizada |
| **ÉTICA** | Transparencia y preocupaciones éticas |

In [ ]:
# =============================================================================
# DEFINICIÓN: MAPEO TECNOLOGÍAS → PREGUNTAS
# =============================================================================

print("=" * 70)
print("ANÁLISIS POR TECNOLOGÍA DE IA")
print("=" * 70)

# Mapeo de tecnologías a preguntas
TECNOLOGIAS = {
    'PA': {
        'nombre': 'People Analytics',
        'descripcion': 'Análisis integral de datos de productividad, rotación, desempeño',
        'preguntas': ['col_ia_detectar_desmotivacion', 'lid_ia_detectar_desmotivacion', 
                      'dir_ger_ia_fuga_talento', 'rrhh_ia_movimientos_internos',
                      'lid_incomodidad_decisiones_auto','dir_ger_ia_deteccion_agotamiento','rrhh_ia_desgaste_emocional']
    },
    'UP/R': {
        'nombre': 'Upskilling/Reskilling',
        'descripcion': 'IA para capacitación y desarrollo',
        'preguntas': ['col_ia_capacitacion', 'lid_ia_capacitacion', 
                      'dir_ger_ia_formacion_personalizada', 'rrhh_ia_brechas_habilidades']
    },
    'CHAT': {
        'nombre': 'Chatbots RRHH',
        'descripcion': 'Asistentes virtuales administrativos',
        'preguntas': ['col_ia_asistente_admin', 'lid_ia_agente_admin', 
                      'dir_ger_ia_asistente_consultas', 'rrhh_ia_asistente_eficiencia']
    },
    'ESA': {
        'nombre': 'Sentiment Analysis',
        'descripcion': 'Análisis de emociones en calls, mensajes, microexpresiones',
        'preguntas': ['col_ia_emociones', 'dir_ger_ia_interpretar_emociones', 
                      'rrhh_ia_analisis_lenguaje', 'lid_ia_expresiones_preocupacion']
    },
    'WELL': {
        'nombre': 'Bienestar Emocional',
        'descripcion': 'Plataformas de bienestar, chat de apoyo, detección de agotamiento',
        'preguntas': ['col_ia_apoyo_emocional', 'col_ia_equilibrio_personal', 
                      'col_ia_asistente_personal', 'lid_ia_alertas_sobrecarga', 
                      'dir_ger_ia_deteccion_agotamiento', 'rrhh_ia_desgaste_emocional','lid_equipo_comodo_asistente']
    },
    'MOB': {
        'nombre': 'Movilidad Interna',
        'descripcion': 'Recomendación de cambios de rol, sucesión',
        'preguntas': ['col_ia_carrera', 'lid_ia_movilidad', 
                      'dir_ger_ia_sucesion', 'rrhh_ia_movimientos_internos']
    },
    'ONA': {
        'nombre': 'Network Analysis',
        'descripcion': 'Análisis de interacciones y redes de colaboración',
        'preguntas': ['col_ia_interacciones', 'lid_ia_comunicacion', 
                      'dir_ger_ia_patrones_comunicacion', 'rrhh_ia_redes_colaboracion']
    },
    'COMP': {
        'nombre': 'Compensación IA',
        'descripcion': 'Sistemas de ajuste salarial basados en IA',
        'preguntas': ['dir_ger_ia_compensaciones', 'rrhh_ia_recomendaciones_salario']
    }
}

# Verificar qué preguntas existen en el dataset
print("\nVerificación de preguntas por tecnología:")
for tech, info in TECNOLOGIAS.items():
    preguntas_existentes = [p for p in info['preguntas'] if p in df_valido.columns]
    print(f"  {tech}: {len(preguntas_existentes)}/{len(info['preguntas'])} preguntas disponibles")

print("\nMapeo de tecnologías definido")

In [ ]:
# =============================================================================
# CÁLCULO: SCORES POR TECNOLOGÍA Y ROL
# =============================================================================

print("=" * 70)
print("CÁLCULO DE SCORES POR TECNOLOGÍA")
print("=" * 70)

# NOTA: Preguntas redactadas en negativo que se invierten (7 - valor)
# para que sean comparables con las demás preguntas de aceptación:
# - lid_ia_expresiones_preocupacion: "podría generar preocupación"
# - lid_incomodidad_decisiones_auto: "Me incomodaría..."
PREGUNTAS_INVERTIR = ['lid_ia_expresiones_preocupacion', 'lid_incomodidad_decisiones_auto']

def calcular_score_tecnologia(df, preguntas):
    """
    Calcula el score promedio de una tecnología para cada persona.
    Solo considera las preguntas que esa persona respondió.
    """
    preguntas_existentes = [p for p in preguntas if p in df.columns]
    if not preguntas_existentes:
        return pd.Series([np.nan] * len(df), index=df.index)
    
    # Crear copia para no modificar original
    datos = df[preguntas_existentes].copy()
    
    # Convertir a numérico
    for col in datos.columns:
        datos[col] = pd.to_numeric(datos[col], errors='coerce')
    
    # Invertir preguntas negativas (7 - valor)
    for col in datos.columns:
        if col in PREGUNTAS_INVERTIR:
            datos[col] = 7 - datos[col]
    
    # Promedio de las preguntas respondidas (ignora NaN)
    return datos.mean(axis=1)

# Calcular score de cada tecnología para cada persona
for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    df_valido[col_name] = calcular_score_tecnologia(df_valido, info['preguntas'])

# Lista de columnas de scores
COLS_SCORES = [f'score_{tech.lower().replace("/", "_")}' for tech in TECNOLOGIAS.keys()]

# Mostrar estadísticas generales
print("\nScores generales por tecnología:")
print(f"{'Tecnología':<15} {'n':>6} {'Media':>7} {'DE':>6} {'Mediana':>8}")
print("-" * 45)

for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    datos = df_valido[col_name].dropna()
    if len(datos) > 0:
        print(f"{tech:<15} {len(datos):>6} {datos.mean():>7.2f} {datos.std():>6.2f} {datos.median():>8.2f}")

print("\nScores calculados para cada tecnología")

In [ ]:
# =============================================================================
# GRÁFICO 1: RANKING GENERAL DE TECNOLOGÍAS
# =============================================================================

print("=" * 70)
print("1. RANKING GENERAL DE ACEPTACIÓN POR TECNOLOGÍA")
print("=" * 70)

# Calcular medias y ordenar
ranking_data = []
for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    datos = df_valido[col_name].dropna()
    if len(datos) > 0:
        ranking_data.append({
            'Tecnología': tech,
            'Nombre': info['nombre'],
            'Media': datos.mean(),
            'DE': datos.std(),
            'n': len(datos)
        })

df_ranking = pd.DataFrame(ranking_data).sort_values('Media', ascending=True)

# Crear gráfico
fig, ax = plt.subplots(figsize=(12, 8))

# Colores según nivel de aceptación
colores = ['#d73027' if m < 3.5 else '#fee090' if m < 4.5 else '#1a9850' 
           for m in df_ranking['Media']]

bars = ax.barh(range(len(df_ranking)), df_ranking['Media'], 
               xerr=df_ranking['DE'], color=colores, edgecolor='white',
               capsize=3, alpha=0.8)

# Etiquetas
ax.set_yticks(range(len(df_ranking)))
ax.set_yticklabels([f"{row['Tecnología']} - {row['Nombre']}" 
                    for _, row in df_ranking.iterrows()])

# Línea de punto medio (3.5)
ax.axvline(x=3.5, color='gray', linestyle='--', alpha=0.7, label='Punto neutro')

# Valores en las barras
for i, (_, row) in enumerate(df_ranking.iterrows()):
    ax.text(row['Media'] + row['DE'] + 0.1, i, 
           f"{row['Media']:.2f} (n={row['n']})", 
           va='center', fontsize=10)

ax.set_xlabel('Nivel de Aceptación (1-6)', fontsize=11)
ax.set_title('Ranking de Aceptación de Tecnologías de IA en RRHH', 
             fontweight='bold', fontsize=14)
ax.set_xlim(1, 6.5)
ax.legend(loc='lower right')

# Leyenda de colores
ax.text(0.02, 0.98, 'Colores: Rojo (<3.5) | Amarillo (3.5-4.5) | Azul (>4.5)', 
        transform=ax.transAxes, fontsize=9, va='top', style='italic')

plt.tight_layout()
plt.savefig('graficos/tech_01_ranking_general.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar tabla
print("\nTabla de ranking:")
display(df_ranking[['Tecnología', 'Nombre', 'Media', 'DE', 'n']].sort_values('Media', ascending=False).round(2))

In [ ]:
# =============================================================================
# GRÁFICO 2: HEATMAP TECNOLOGÍA x ROL
# =============================================================================

print("=" * 70)
print("2. HEATMAP: ACEPTACIÓN POR TECNOLOGÍA Y ROL")
print("=" * 70)

# Calcular medias por rol y tecnología
orden_roles = ['Colaborador', 'Líder', 'Dirección/Gerencia', 'RRHH']

heatmap_data = []
for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    for rol in orden_roles:
        datos = df_valido[df_valido['rol'] == rol][col_name].dropna()
        if len(datos) > 0:
            heatmap_data.append({
                'Tecnología': tech,
                'Rol': rol,
                'Media': datos.mean(),
                'n': len(datos)
            })

df_heatmap = pd.DataFrame(heatmap_data)

# Pivotear para heatmap
matriz_heatmap = df_heatmap.pivot(index='Tecnología', columns='Rol', values='Media')
matriz_heatmap = matriz_heatmap[orden_roles]  # Ordenar columnas

# Ordenar filas por promedio general
matriz_heatmap['_promedio'] = matriz_heatmap.mean(axis=1)
matriz_heatmap = matriz_heatmap.sort_values('_promedio', ascending=False)
matriz_heatmap = matriz_heatmap.drop('_promedio', axis=1)

# Crear heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(matriz_heatmap, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=3.5, vmin=1, vmax=6, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Nivel de Aceptación'})

ax.set_title('Aceptación de Tecnologías de IA por Rol', fontweight='bold', fontsize=14)
ax.set_xlabel('Rol')
ax.set_ylabel('Tecnología')

plt.tight_layout()
plt.savefig('graficos/tech_02_heatmap_rol.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar tabla con n por celda
print("\nCantidad de respuestas por celda:")
matriz_n = df_heatmap.pivot(index='Tecnología', columns='Rol', values='n')
matriz_n = matriz_n[orden_roles]
display(matriz_n)

In [ ]:
# =============================================================================
# GRÁFICO 3: COMPARACIÓN DETALLADA POR ROL
# =============================================================================

print("=" * 70)
print("3. COMPARACIÓN DETALLADA POR ROL")
print("=" * 70)

# Crear gráfico de barras agrupadas
orden_roles = ['Colaborador', 'Líder', 'Dirección/Gerencia', 'RRHH']
techs = list(TECNOLOGIAS.keys())

fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(techs))
width = 0.2

for i, rol in enumerate(orden_roles):
    medias = []
    for tech in techs:
        col_name = f'score_{tech.lower().replace("/", "_")}'
        datos = df_valido[df_valido['rol'] == rol][col_name].dropna()
        medias.append(datos.mean() if len(datos) > 0 else np.nan)
    
    bars = ax.bar(x + i * width, medias, width, 
                  label=rol, color=COLORES_ROLES.get(rol, '#999999'),
                  edgecolor='white')

# Configuración
ax.set_xlabel('Tecnología', fontsize=11)
ax.set_ylabel('Nivel de Aceptación (1-6)', fontsize=11)
ax.set_title('Aceptación de Tecnologías de IA por Rol', fontweight='bold', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(techs, rotation=45, ha='right')
ax.legend(title='Rol', loc='upper right')
ax.set_ylim(1, 6)
ax.axhline(y=3.5, color='gray', linestyle='--', alpha=0.5, label='Punto neutro')

# Añadir nombres completos abajo
nombres_tech = [TECNOLOGIAS[t]['nombre'] for t in techs]
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(x + width * 1.5)
ax2.set_xticklabels(nombres_tech, fontsize=8, rotation=45, ha='left')

plt.tight_layout()
plt.savefig('graficos/tech_03_comparacion_rol.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# GRÁFICO 4: CORRELACIÓN TECNOLOGÍAS vs VARIABLES DE PERCEPCIÓN (Escala 1-6)
# =============================================================================

print("=" * 70)
print("4. CORRELACIÓN: TECNOLOGÍAS vs VARIABLES DE PERCEPCIÓN")
print("=" * 70)
print("\n(Variables en escala 1-6, aptas para correlación de Spearman)")

from scipy.stats import spearmanr

# Variables de escala 1-6 (aptas para correlación)
vars_escala = {
    'ia_positiva': 'IA Positiva',
    'confianza_ia': 'Confianza IA',
    'ia_invasivo': 'IA Invasivo'
}

# Calcular correlaciones
corr_data = []
for tech in TECNOLOGIAS.keys():
    col_score = f'score_{tech.lower().replace("/", "_")}'
    if col_score not in df_valido.columns:
        continue
    
    for var, nombre_var in vars_escala.items():
        if var not in df_valido.columns:
            continue
        
        datos = df_valido[[col_score, var]].copy()
        datos[col_score] = pd.to_numeric(datos[col_score], errors='coerce')
        datos[var] = pd.to_numeric(datos[var], errors='coerce')
        datos = datos.dropna()
        
        if len(datos) > 10:
            corr, pval = spearmanr(datos[col_score], datos[var])
            corr_data.append({
                'Tecnología': tech,
                'Variable': nombre_var,
                'Correlación': corr,
                'p-valor': pval,
                'n': len(datos)
            })

df_corr_tech = pd.DataFrame(corr_data)

# Crear matriz para heatmap
matriz_corr = df_corr_tech.pivot(index='Tecnología', columns='Variable', values='Correlación')

# Crear heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=0, vmin=-0.5, vmax=0.5, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Correlación de Spearman'})

ax.set_title('Correlación: Tecnologías vs Variables de Percepción\n(Escala 1-6)', fontweight='bold', fontsize=14)
ax.set_xlabel('Variable')
ax.set_ylabel('Tecnología')

plt.tight_layout()
plt.savefig('graficos/tech_04_correlaciones_escala.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar correlaciones significativas
print("\nCorrelaciones significativas (p < 0.05):")
df_sig = df_corr_tech[df_corr_tech['p-valor'] < 0.05].sort_values('Correlación', key=abs, ascending=False)
if len(df_sig) > 0:
    display(df_sig.head(15).round(3))
else:
    print("No se encontraron correlaciones significativas")

In [ ]:
# =============================================================================
# GRÁFICO 4.1: ACEPTACIÓN POR VARIABLES DEMOGRÁFICAS (Comparación de Medias)
# =============================================================================

print("=" * 70)
print("4.1 ACEPTACIÓN DE TECNOLOGÍAS POR VARIABLES DEMOGRÁFICAS")
print("=" * 70)
print("\n(Comparación de medias por grupo - responde: ¿Quién acepta más?)")

from scipy.stats import kruskal

# --- ANÁLISIS POR EDAD ---
print("\n" + "=" * 50)
print("A) ACEPTACIÓN POR GRUPO DE EDAD")
print("=" * 50)

# Orden de grupos de edad
orden_edad = ['Menos de 25 años', 'Entre 25 y 35 años', 'Entre 36 y 45 años', 
              'Entre 46 y 55 años', 'Mayor de 55 años']

# Calcular medias por edad para cada tecnología
resultados_edad = []
for tech in TECNOLOGIAS.keys():
    col_score = f'score_{tech.lower().replace("/", "_")}'
    if col_score not in df_valido.columns:
        continue
    
    for edad in orden_edad:
        datos = df_valido[df_valido['edad'] == edad][col_score].dropna()
        if len(datos) > 0:
            resultados_edad.append({
                'Tecnología': tech,
                'Edad': edad,
                'Media': datos.mean(),
                'n': len(datos)
            })

df_edad = pd.DataFrame(resultados_edad)

# Pivotear para tabla
tabla_edad = df_edad.pivot(index='Tecnología', columns='Edad', values='Media')
tabla_edad = tabla_edad[orden_edad]  # Ordenar columnas
print("\nMedia de aceptación por grupo de edad:")
display(tabla_edad.round(2))

# Heatmap por edad
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(tabla_edad, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=3.5, vmin=2, vmax=6, ax=ax, linewidths=0.5)
ax.set_title('Aceptación de Tecnologías por Grupo de Edad', fontweight='bold', fontsize=14)
ax.set_xlabel('Grupo de Edad')
ax.set_ylabel('Tecnología')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('graficos/tech_04b_por_edad.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# --- ANÁLISIS POR ANTIGÜEDAD ---
print("\n" + "=" * 50)
print("B) ACEPTACIÓN POR ANTIGÜEDAD EN LA EMPRESA")
print("=" * 50)

# Orden de grupos de antigüedad
orden_antiguedad = ['De 0 a 2 años', 'De 2 a 5 años', 'De 5 a 15 años', 'Más de 15 años']

# Calcular medias por antigüedad para cada tecnología
resultados_ant = []
for tech in TECNOLOGIAS.keys():
    col_score = f'score_{tech.lower().replace("/", "_")}'
    if col_score not in df_valido.columns:
        continue
    
    for ant in orden_antiguedad:
        datos = df_valido[df_valido['antiguedad'] == ant][col_score].dropna()
        if len(datos) > 0:
            resultados_ant.append({
                'Tecnología': tech,
                'Antigüedad': ant,
                'Media': datos.mean(),
                'n': len(datos)
            })

df_ant = pd.DataFrame(resultados_ant)

# Pivotear para tabla
tabla_ant = df_ant.pivot(index='Tecnología', columns='Antigüedad', values='Media')
tabla_ant = tabla_ant[orden_antiguedad]  # Ordenar columnas
print("\nMedia de aceptación por antigüedad:")
display(tabla_ant.round(2))

# Heatmap por antigüedad
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(tabla_ant, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=3.5, vmin=2, vmax=6, ax=ax, linewidths=0.5)
ax.set_title('Aceptación de Tecnologías por Antigüedad', fontweight='bold', fontsize=14)
ax.set_xlabel('Antigüedad en la Empresa')
ax.set_ylabel('Tecnología')
plt.tight_layout()
plt.savefig('graficos/tech_04c_por_antiguedad.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# --- ANÁLISIS POR TAMAÑO DE EMPRESA ---
print("\n" + "=" * 50)
print("C) ACEPTACIÓN POR TAMAÑO DE EMPRESA")
print("=" * 50)

# Orden de grupos de tamaño
orden_tamanio = ['Menos de 50 empleados', 'De 50 a 100 empleados', 
                 'De 100 a 300 empleados', 'Más de 300 empleados']

# Calcular medias por tamaño para cada tecnología
resultados_tam = []
for tech in TECNOLOGIAS.keys():
    col_score = f'score_{tech.lower().replace("/", "_")}'
    if col_score not in df_valido.columns:
        continue
    
    for tam in orden_tamanio:
        datos = df_valido[df_valido['tamanio_empresa_global'] == tam][col_score].dropna()
        if len(datos) > 0:
            resultados_tam.append({
                'Tecnología': tech,
                'Tamaño': tam,
                'Media': datos.mean(),
                'n': len(datos)
            })

df_tam = pd.DataFrame(resultados_tam)

# Pivotear para tabla
tabla_tam = df_tam.pivot(index='Tecnología', columns='Tamaño', values='Media')
tabla_tam = tabla_tam[orden_tamanio]  # Ordenar columnas
print("\nMedia de aceptación por tamaño de empresa:")
display(tabla_tam.round(2))

# Heatmap por tamaño
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(tabla_tam, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=3.5, vmin=2, vmax=6, ax=ax, linewidths=0.5)
ax.set_title('Aceptación de Tecnologías por Tamaño de Empresa', fontweight='bold', fontsize=14)
ax.set_xlabel('Tamaño de Empresa')
ax.set_ylabel('Tecnología')
plt.tight_layout()
plt.savefig('graficos/tech_04d_por_tamanio.png', dpi=150, bbox_inches='tight')
plt.show()




In [ ]:
# =============================================================================
# GRÁFICO 5: ANÁLISIS DE BRECHAS ENTRE ROLES
# =============================================================================

print("=" * 70)
print("5. ANÁLISIS DE BRECHAS: DIFERENCIAS ENTRE ROLES")
print("=" * 70)

# Calcular brecha (diferencia max - min) por tecnología
orden_roles = ['Colaborador', 'Líder', 'Dirección/Gerencia', 'RRHH']

brechas_data = []
for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    
    medias_rol = {}
    for rol in orden_roles:
        datos = df_valido[df_valido['rol'] == rol][col_name].dropna()
        if len(datos) > 0:
            medias_rol[rol] = datos.mean()
    
    if len(medias_rol) >= 2:
        max_val = max(medias_rol.values())
        min_val = min(medias_rol.values())
        rol_max = [k for k, v in medias_rol.items() if v == max_val][0]
        rol_min = [k for k, v in medias_rol.items() if v == min_val][0]
        
        brechas_data.append({
            'Tecnología': tech,
            'Nombre': info['nombre'],
            'Brecha': max_val - min_val,
            'Rol_Mayor': rol_max,
            'Valor_Mayor': max_val,
            'Rol_Menor': rol_min,
            'Valor_Menor': min_val
        })

df_brechas = pd.DataFrame(brechas_data).sort_values('Brecha', ascending=False)

# Gráfico de brechas
fig, ax = plt.subplots(figsize=(12, 8))

y_pos = range(len(df_brechas))

# Barras para valores mínimos y máximos
for i, (_, row) in enumerate(df_brechas.iterrows()):
    # Línea de brecha
    ax.plot([row['Valor_Menor'], row['Valor_Mayor']], [i, i], 
            color='gray', linewidth=2, zorder=1)
    
    # Punto mínimo
    ax.scatter(row['Valor_Menor'], i, s=150, 
               color=COLORES_ROLES.get(row['Rol_Menor'], '#999999'),
               zorder=2, edgecolor='white', linewidth=2)
    
    # Punto máximo
    ax.scatter(row['Valor_Mayor'], i, s=150, 
               color=COLORES_ROLES.get(row['Rol_Mayor'], '#999999'),
               zorder=2, edgecolor='white', linewidth=2)
    
    # Etiqueta de brecha
    ax.text(row['Valor_Mayor'] + 0.1, i, f"Δ={row['Brecha']:.2f}", 
            va='center', fontsize=9, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row['Tecnología']} - {row['Nombre']}" 
                    for _, row in df_brechas.iterrows()])
ax.set_xlabel('Nivel de Aceptación (1-6)', fontsize=11)
ax.set_title('Brechas de Aceptación entre Roles por Tecnología', fontweight='bold', fontsize=14)
ax.set_xlim(1, 6.5)
ax.axvline(x=3.5, color='gray', linestyle='--', alpha=0.3)

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=rol) 
                   for rol, color in COLORES_ROLES.items()]
ax.legend(handles=legend_elements, title='Rol', loc='lower right')

plt.tight_layout()
plt.savefig('graficos/tech_05_brechas.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla de brechas
print("\nTabla de brechas (ordenada de mayor a menor):")
display(df_brechas[['Tecnología', 'Brecha', 'Rol_Mayor', 'Valor_Mayor', 'Rol_Menor', 'Valor_Menor']].round(2))

In [ ]:
# =============================================================================
# GRÁFICO 6: ANÁLISIS POR MODALIDAD DE TRABAJO
# =============================================================================

print("=" * 70)
print("6. ACEPTACIÓN DE TECNOLOGÍAS POR MODALIDAD DE TRABAJO")
print("=" * 70)

# Orden de modalidades
orden_modalidades = ['Presencial', 'Híbrido', 'Remoto']

# Colores para modalidades
colores_modalidad = {
    'Presencial': '#2E86AB',
    'Híbrido': '#A23B72', 
    'Remoto': '#F18F01'
}

# Calcular scores promedio por modalidad para cada tecnología
scores_modalidad = []

for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    
    for modalidad in orden_modalidades:
        datos = df_valido[df_valido['modalidad_trabajo'] == modalidad][col_name].dropna()
        if len(datos) > 0:
            scores_modalidad.append({
                'Tecnología': tech,
                'Nombre': info['nombre'],
                'Modalidad': modalidad,
                'Media': datos.mean(),
                'DE': datos.std(),
                'n': len(datos)
            })

df_modalidad = pd.DataFrame(scores_modalidad)

# Tabla resumen
print("\nScores promedio por modalidad de trabajo:")
print("-" * 70)
pivot_modalidad = df_modalidad.pivot_table(
    values='Media', 
    index='Tecnología', 
    columns='Modalidad'
)[orden_modalidades]
print(pivot_modalidad.round(2).to_string())

# Gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=(14, 8))

techs = list(TECNOLOGIAS.keys())
x = np.arange(len(techs))
width = 0.25

for i, modalidad in enumerate(orden_modalidades):
    df_mod = df_modalidad[df_modalidad['Modalidad'] == modalidad]
    medias = [df_mod[df_mod['Tecnología'] == tech]['Media'].values[0] 
              if len(df_mod[df_mod['Tecnología'] == tech]) > 0 else 0 
              for tech in techs]
    bars = ax.bar(x + i*width, medias, width, label=modalidad, 
                  color=colores_modalidad[modalidad], alpha=0.85)

ax.set_xlabel('Tecnología', fontsize=12)
ax.set_ylabel('Score de Aceptación (1-6)', fontsize=12)
ax.set_title('Aceptación de Tecnologías de IA por Modalidad de Trabajo', 
             fontweight='bold', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels([TECNOLOGIAS[t]['nombre'] for t in techs], rotation=45, ha='right')
ax.legend(title='Modalidad')
ax.set_ylim(0, 6)
ax.axhline(y=3.5, color='gray', linestyle='--', alpha=0.5, label='Punto medio')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Análisis estadístico: Kruskal-Wallis por tecnología
print("\n" + "=" * 70)
print("ANÁLISIS ESTADÍSTICO: DIFERENCIAS POR MODALIDAD (Kruskal-Wallis)")
print("=" * 70)

for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    
    grupos = []
    for modalidad in orden_modalidades:
        datos = df_valido[df_valido['modalidad_trabajo'] == modalidad][col_name].dropna()
        if len(datos) >= 3:
            grupos.append(datos.values)
    
    if len(grupos) >= 2:
        stat, p_value = kruskal(*grupos)
        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
        print(f"{info['nombre']:<30} H={stat:>7.2f}  p={p_value:.4f} {sig}")

print("\nSignificancia: * p<0.05, ** p<0.01, *** p<0.001")

# Heatmap de diferencias por modalidad
print("\n" + "=" * 70)
print("HEATMAP: SCORES POR TECNOLOGÍA Y MODALIDAD")
print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 8))

# Preparar datos para heatmap
heatmap_data = pivot_modalidad.values
techs_nombres = [TECNOLOGIAS[t]['nombre'] for t in pivot_modalidad.index]

im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=1, vmax=6)

ax.set_xticks(np.arange(len(orden_modalidades)))
ax.set_yticks(np.arange(len(techs_nombres)))
ax.set_xticklabels(orden_modalidades)
ax.set_yticklabels(techs_nombres)

# Añadir valores
for i in range(len(techs_nombres)):
    for j in range(len(orden_modalidades)):
        valor = heatmap_data[i, j]
        color = 'white' if valor < 2.5 or valor > 4.5 else 'black'
        ax.text(j, i, f'{valor:.2f}', ha='center', va='center', color=color, fontsize=11)

ax.set_title('Aceptación por Tecnología y Modalidad de Trabajo\n(Verde=Mayor aceptación, Rojo=Menor)', 
             fontweight='bold', fontsize=12)
plt.colorbar(im, ax=ax, label='Score (1-6)')

plt.tight_layout()
plt.show()



In [ ]:
# =============================================================================
# GRÁFICO 7: ANÁLISIS POR GÉNERO (Hombre vs Mujer)
# =============================================================================

print("=" * 70)
print("7. ACEPTACIÓN DE TECNOLOGÍAS POR GÉNERO")
print("=" * 70)

# Filtrar solo Hombre y Mujer
df_genero = df_valido[df_valido['genero'].isin(['Hombre', 'Mujer'])].copy()
print(f"\nMuestra filtrada: {len(df_genero)} respuestas (Hombres: {len(df_genero[df_genero['genero']=='Hombre'])}, Mujeres: {len(df_genero[df_genero['genero']=='Mujer'])})")

# Orden de géneros
orden_generos = ['Hombre', 'Mujer']

# Colores para géneros
colores_genero = {
    'Hombre': '#2E86AB',
    'Mujer': '#A23B72'
}

# Calcular scores promedio por género para cada tecnología
scores_genero = []

for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    
    for genero in orden_generos:
        datos = df_genero[df_genero['genero'] == genero][col_name].dropna()
        if len(datos) > 0:
            scores_genero.append({
                'Tecnología': tech,
                'Nombre': info['nombre'],
                'Género': genero,
                'Media': datos.mean(),
                'DE': datos.std(),
                'n': len(datos)
            })

df_scores_genero = pd.DataFrame(scores_genero)

# Tabla resumen
print("\nScores promedio por género:")
print("-" * 50)
pivot_genero = df_scores_genero.pivot_table(
    values='Media', 
    index='Tecnología', 
    columns='Género'
)[orden_generos]
print(pivot_genero.round(2).to_string())

# Calcular diferencias
print("\nDiferencia (Mujer - Hombre):")
print("-" * 50)
for tech in pivot_genero.index:
    diff = pivot_genero.loc[tech, 'Mujer'] - pivot_genero.loc[tech, 'Hombre']
    signo = "+" if diff > 0 else ""
    print(f"{TECNOLOGIAS[tech]['nombre']:<30} {signo}{diff:.2f}")

# Gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=(14, 8))

techs = list(TECNOLOGIAS.keys())
x = np.arange(len(techs))
width = 0.35

for i, genero in enumerate(orden_generos):
    df_gen = df_scores_genero[df_scores_genero['Género'] == genero]
    medias = [df_gen[df_gen['Tecnología'] == tech]['Media'].values[0] 
              if len(df_gen[df_gen['Tecnología'] == tech]) > 0 else 0 
              for tech in techs]
    bars = ax.bar(x + i*width, medias, width, label=genero, 
                  color=colores_genero[genero], alpha=0.85)

ax.set_xlabel('Tecnología', fontsize=12)
ax.set_ylabel('Score de Aceptación (1-6)', fontsize=12)
ax.set_title('Aceptación de Tecnologías de IA por Género', 
             fontweight='bold', fontsize=14)
ax.set_xticks(x + width/2)
ax.set_xticklabels([TECNOLOGIAS[t]['nombre'] for t in techs], rotation=45, ha='right')
ax.legend(title='Género')
ax.set_ylim(0, 6)
ax.axhline(y=3.5, color='gray', linestyle='--', alpha=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Análisis estadístico: Mann-Whitney U por tecnología
print("\n" + "=" * 70)
print("ANÁLISIS ESTADÍSTICO: DIFERENCIAS POR GÉNERO (Mann-Whitney U)")
print("=" * 70)

from scipy.stats import mannwhitneyu

for tech, info in TECNOLOGIAS.items():
    col_name = f'score_{tech.lower().replace("/", "_")}'
    
    hombres = df_genero[df_genero['genero'] == 'Hombre'][col_name].dropna()
    mujeres = df_genero[df_genero['genero'] == 'Mujer'][col_name].dropna()
    
    if len(hombres) >= 3 and len(mujeres) >= 3:
        stat, p_value = mannwhitneyu(hombres, mujeres, alternative='two-sided')
        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
        print(f"{info['nombre']:<30} U={stat:>8.1f}  p={p_value:.4f} {sig}")

print("\nSignificancia: * p<0.05, ** p<0.01, *** p<0.001")

# Gráfico de diferencias (lollipop chart)
print("\n" + "=" * 70)
print("GRÁFICO DE DIFERENCIAS POR GÉNERO")
print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 8))

diferencias = []
for tech in TECNOLOGIAS.keys():
    diff = pivot_genero.loc[tech, 'Mujer'] - pivot_genero.loc[tech, 'Hombre']
    diferencias.append({
        'Tecnología': TECNOLOGIAS[tech]['nombre'],
        'Diferencia': diff
    })

df_diff = pd.DataFrame(diferencias).sort_values('Diferencia')

colors = ['#A23B72' if x > 0 else '#2E86AB' for x in df_diff['Diferencia']]
ax.barh(df_diff['Tecnología'], df_diff['Diferencia'], color=colors, alpha=0.8)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Diferencia en Score (Mujer - Hombre)', fontsize=12)
ax.set_title('Diferencias de Aceptación por Género\n(Positivo = Mujeres más favorables, Negativo = Hombres más favorables)', 
             fontweight='bold', fontsize=12)
ax.grid(axis='x', alpha=0.3)

# Añadir valores
for i, (_, row) in enumerate(df_diff.iterrows()):
    ax.text(row['Diferencia'] + 0.02 if row['Diferencia'] >= 0 else row['Diferencia'] - 0.02, 
            i, f"{row['Diferencia']:+.2f}", va='center', 
            ha='left' if row['Diferencia'] >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()



---
## 9. Correlaciones

In [ ]:
# =============================================================================
# GRÁFICO 6: CLUSTER DE TECNOLOGÍAS (¿CUÁLES VAN JUNTAS?)
# =============================================================================

print("=" * 70)
print("6. CLUSTER: ¿QUÉ TECNOLOGÍAS VAN JUNTAS?")
print("=" * 70)

from scipy.stats import spearmanr

# Crear matriz de correlación entre tecnologías
techs = list(TECNOLOGIAS.keys())
cols_scores = [f'score_{t.lower().replace("/", "_")}' for t in techs]
cols_existentes = [c for c in cols_scores if c in df_valido.columns]

# Calcular correlaciones entre tecnologías
matriz_tech = df_valido[cols_existentes].corr(method='spearman')

# Renombrar para visualización
nombres_tech = {f'score_{t.lower().replace("/", "_")}': t for t in techs}
matriz_tech = matriz_tech.rename(index=nombres_tech, columns=nombres_tech)

# Crear heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Mask para triangular superior
mask = np.triu(np.ones_like(matriz_tech, dtype=bool), k=1)

sns.heatmap(matriz_tech, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=0.5, vmin=0, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Correlación de Spearman'})

ax.set_title('Correlación entre Tecnologías\n(¿Quién acepta una, acepta la otra?)', 
             fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig('graficos/tech_06_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar pares con mayor correlación
print("\nPares de tecnologías más correlacionadas (r > 0.5):")
correlaciones_pares = []
for i, tech1 in enumerate(matriz_tech.columns):
    for j, tech2 in enumerate(matriz_tech.columns):
        if i < j:
            corr = matriz_tech.loc[tech1, tech2]
            if corr > 0.5:
                correlaciones_pares.append({
                    'Tecnología 1': tech1,
                    'Tecnología 2': tech2,
                    'Correlación': corr
                })

if correlaciones_pares:
    df_pares = pd.DataFrame(correlaciones_pares).sort_values('Correlación', ascending=False)
    display(df_pares.round(2))
else:
    print("No se encontraron pares con correlación > 0.5")

print("\nInterpretación:")
print("   - Correlación alta (verde): Quien acepta una tecnología, tiende a aceptar la otra")
print("   - Correlación baja (rojo): La aceptación de una no predice la aceptación de la otra")